## Этап 1. Загрузка исходных данных
Проверяем источник GDB, поднимаем кэш и делаем первичный обзор таблицы.

In [1]:
import geopandas as gpd

### 1.1 Источник геоданных
Задаем путь к geodatabase и смотрим доступные слои.

In [2]:
gdb_path = r"C:\Users\Dmitry\Downloads\NationalCSB_2017-2024_rev23\CSB1724.gdb"
layers = gpd.list_layers(gdb_path)

In [3]:
from pathlib import Path

layer_name = layers.iloc[0]["name"]
cache_path = Path("national1724.pkl")
print("Layer:", layer_name)
print("Cache:", cache_path.resolve())

Layer: national1724
Cache: C:\Users\Dmitry\code-projects\diploma-crop-rotation\national1724.pkl


### 1.2 Кэш и чтение слоя
Если кэш уже есть, читаем pickle; иначе загружаем слой из GDB и сохраняем кэш.

In [4]:
import pandas as pd
from pyogrio import read_dataframe

if cache_path.exists():
    df = pd.read_pickle(cache_path)
    print(f"Loaded cached pickle: {cache_path}")
else:
    df = read_dataframe(
        gdb_path,
        layer=layer_name,
        read_geometry=False
    )
    df.to_pickle(cache_path)
    print(f"Saved pickle cache: {cache_path}")

df.head()

Loaded cached pickle: national1724.pkl


,CSBID,CSBYEARS,CSBACRES,CDL2017,CDL2018,CDL2019,CDL2020,CDL2021,CDL2022,CDL2023,CDL2024,STATEFIPS,STATEASD,ASD,CNTY,CNTYFIPS,INSIDE_X,INSIDE_Y,Shape_Length,Shape_Area
0,351724000000001,1724,2.777993,24,24,152,152,152,152,152,152,35,3530,30,Union,059,-650336.7078,1.447441e+06,628.315309,11242.149046
1,351724000000002,1724,9.293377,24,24,24,24,24,24,24,152,35,3530,30,Union,059,-648203.5992,1.447142e+06,1042.818836,37608.996229
2,351724000000003,1724,5.775892,24,24,24,24,24,24,24,152,35,3530,30,Union,059,-647571.4914,1.447064e+06,1164.537470,23374.226788
3,351724000000004,1724,5.206396,24,24,27,236,24,24,24,152,35,3530,30,Union,059,-647642.6595,1.446862e+06,760.543172,21069.554571
4,351724000000005,1724,50.284333,24,24,27,236,24,24,24,152,35,3530,30,Union,059,-647922.0470,1.447090e+06,2879.640092,203493.655293


### 1.3 Первичная проверка данных
Фиксируем размер таблицы, состав колонок и небольшой preview.

In [5]:
print(f"Dataset shape: {df.shape}")
print("Columns:", df.columns.tolist())
df.head(3)

Dataset shape: (16418258, 20)
Columns: ['CSBID', 'CSBYEARS', 'CSBACRES', 'CDL2017', 'CDL2018', 'CDL2019', 'CDL2020', 'CDL2021', 'CDL2022', 'CDL2023', 'CDL2024', 'STATEFIPS', 'STATEASD', 'ASD', 'CNTY', 'CNTYFIPS', 'INSIDE_X', 'INSIDE_Y', 'Shape_Length', 'Shape_Area']


,CSBID,CSBYEARS,CSBACRES,CDL2017,CDL2018,CDL2019,CDL2020,CDL2021,CDL2022,CDL2023,CDL2024,STATEFIPS,STATEASD,ASD,CNTY,CNTYFIPS,INSIDE_X,INSIDE_Y,Shape_Length,Shape_Area
0,351724000000001,1724,2.777993,24,24,152,152,152,152,152,152,35,3530,30,Union,059,-650336.7078,1.447441e+06,628.315309,11242.149046
1,351724000000002,1724,9.293377,24,24,24,24,24,24,24,152,35,3530,30,Union,059,-648203.5992,1.447142e+06,1042.818836,37608.996229
2,351724000000003,1724,5.775892,24,24,24,24,24,24,24,152,35,3530,30,Union,059,-647571.4914,1.447064e+06,1164.537470,23374.226788


## Этап 2. Справочники и правила классов
Определяем словари агрегации CDL и наборы классов для дальнейшей очистки.

### 2.1 Агрегация кодов CDL в укрупненные группы
Словарь используется для создания колонок вида CDL{year}_group.

In [6]:
CDL_AGGREGATION = {
    1: "corn",
    2: "cotton",
    3: "rice",
    4: "sorghum",
    5: "soybeans",
    6: "sunflower",

    10: "peanuts",
    11: "specialty_crops",   # Tobacco
    12: "specialty_crops",   # Sweet Corn
    13: "specialty_crops",   # Pop or Orn Corn
    14: "specialty_crops",   # Mint

    21: "other_cereals",     # Barley
    22: "wheat",             # Durum Wheat
    23: "wheat",             # Spring Wheat
    24: "wheat",             # Winter Wheat
    25: "other_cereals",     # Other Small Grains
    26: "double_crop",       # Dbl Crop WinWht/Soybeans
    27: "other_cereals",     # Rye
    28: "other_cereals",     # Oats
    29: "other_cereals",     # Millet
    30: "other_cereals",     # Speltz

    31: "oilseeds_other",    # Canola
    32: "oilseeds_other",    # Flaxseed
    33: "oilseeds_other",    # Safflower
    34: "oilseeds_other",    # Rape Seed
    35: "oilseeds_other",    # Mustard
    36: "forage_hay",        # Alfalfa
    37: "forage_hay",        # Other Hay/Non Alfalfa
    38: "oilseeds_other",    # Camelina
    39: "specialty_crops",   # Buckwheat

    41: "sugar_crops",       # Sugarbeets
    42: "legumes",           # Dry Beans
    43: "root_tubers",       # Potatoes
    44: "specialty_crops",   # Other Crops
    45: "sugar_crops",       # Sugarcane
    46: "root_tubers",       # Sweet Potatoes
    47: "vegetables_melons", # Misc Vegs & Fruits
    48: "vegetables_melons", # Watermelons
    49: "vegetables_melons", # Onions
    50: "vegetables_melons", # Cucumbers
    51: "legumes",           # Chick Peas
    52: "legumes",           # Lentils
    53: "legumes",           # Peas
    54: "vegetables_melons", # Tomatoes
    55: "fruits_berries",    # Caneberries
    56: "specialty_crops",   # Hops
    57: "specialty_crops",   # Herbs
    58: "forage_hay",        # Clover/Wildflowers
    59: "forage_hay",        # Sod/Grass Seed
    60: "forage_hay",        # Switchgrass
    61: "fallow",            # Fallow/Idle Cropland
    62: "pasture_grass",     # Pasture/Grass
    63: "non_ag_natural",    # Forest
    64: "non_ag_natural",    # Shrubland
    65: "non_ag_natural",    # Barren
    66: "orchards_vineyards_tree_crops",  # Cherries
    67: "orchards_vineyards_tree_crops",  # Peaches
    68: "orchards_vineyards_tree_crops",  # Apples
    69: "orchards_vineyards_tree_crops",  # Grapes
    70: "orchards_vineyards_tree_crops",  # Christmas Trees
    71: "orchards_vineyards_tree_crops",  # Other Tree Crops
    72: "orchards_vineyards_tree_crops",  # Citrus
    74: "orchards_vineyards_tree_crops",  # Pecans
    75: "orchards_vineyards_tree_crops",  # Almonds
    76: "orchards_vineyards_tree_crops",  # Walnuts
    77: "orchards_vineyards_tree_crops",  # Pears

    81: "developed_water_wetlands",  # Clouds/No Data
    82: "developed_water_wetlands",  # Developed
    83: "developed_water_wetlands",  # Water
    87: "developed_water_wetlands",  # Wetlands
    88: "developed_water_wetlands",  # Nonag/Undefined
    92: "developed_water_wetlands",  # Aquaculture

    111: "developed_water_wetlands", # Open Water
    112: "developed_water_wetlands", # Perennial Ice/Snow
    121: "developed_water_wetlands", # Developed/Open Space
    122: "developed_water_wetlands", # Developed/Low Intensity
    123: "developed_water_wetlands", # Developed/Med Intensity
    124: "developed_water_wetlands", # Developed/High Intensity
    131: "non_ag_natural",           # Barren
    141: "non_ag_natural",           # Deciduous Forest
    142: "non_ag_natural",           # Evergreen Forest
    143: "non_ag_natural",           # Mixed Forest
    152: "non_ag_natural",           # Shrubland
    176: "pasture_grass",            # Grassland/Pasture
    190: "developed_water_wetlands", # Woody Wetlands
    195: "developed_water_wetlands", # Herbaceous Wetlands

    204: "orchards_vineyards_tree_crops",  # Pistachios
    205: "other_cereals",                  # Triticale
    206: "vegetables_melons",             # Carrots
    207: "vegetables_melons",             # Asparagus
    208: "root_tubers",                   # Garlic
    209: "vegetables_melons",             # Cantaloupes
    210: "orchards_vineyards_tree_crops", # Prunes
    211: "orchards_vineyards_tree_crops", # Olives
    212: "orchards_vineyards_tree_crops", # Oranges
    213: "vegetables_melons",             # Honeydew Melons
    214: "vegetables_melons",             # Broccoli
    215: "orchards_vineyards_tree_crops", # Avocados
    216: "vegetables_melons",             # Peppers
    217: "orchards_vineyards_tree_crops", # Pomegranates
    218: "orchards_vineyards_tree_crops", # Nectarines
    219: "vegetables_melons",             # Greens
    220: "orchards_vineyards_tree_crops", # Plums
    221: "fruits_berries",                # Strawberries
    222: "vegetables_melons",             # Squash
    223: "orchards_vineyards_tree_crops", # Apricots
    224: "legumes",                       # Vetch
    225: "double_crop",                   # Dbl Crop WinWht/Corn
    226: "double_crop",                   # Dbl Crop Oats/Corn
    227: "vegetables_melons",             # Lettuce
    228: "double_crop",                   # Dbl Crop Triticale/Corn
    229: "vegetables_melons",             # Pumpkins
    230: "double_crop",                   # Dbl Crop Lettuce/Durum Wht
    231: "double_crop",                   # Dbl Crop Lettuce/Cantaloupe
    232: "double_crop",                   # Dbl Crop Lettuce/Cotton
    233: "double_crop",                   # Dbl Crop Lettuce/Barley
    234: "double_crop",                   # Dbl Crop Durum Wht/Sorghum
    235: "double_crop",                   # Dbl Crop Barley/Sorghum
    236: "double_crop",                   # Dbl Crop WinWht/Sorghum
    237: "double_crop",                   # Dbl Crop Barley/Corn
    238: "double_crop",                   # Dbl Crop WinWht/Cotton
    239: "double_crop",                   # Dbl Crop Soybeans/Cotton
    240: "double_crop",                   # Dbl Crop Soybeans/Oats
    241: "double_crop",                   # Dbl Crop Corn/Soybeans
    242: "fruits_berries",                # Blueberries
    243: "vegetables_melons",             # Cabbage
    244: "vegetables_melons",             # Cauliflower
    245: "vegetables_melons",             # Celery
    246: "root_tubers",                   # Radishes
    247: "root_tubers",                   # Turnips
    248: "vegetables_melons",             # Eggplants
    249: "vegetables_melons",             # Gourds
    250: "fruits_berries",                # Cranberries
    254: "double_crop",                   # Dbl Crop Barley/Soybeans
}

### 2.2 Классы для keep/drop
Фиксируем основные целевые группы и нежелательные классы для фильтрации.

In [7]:
KEEP_MAIN_CLASSES = {
    "corn",
    "soybeans",
    "wheat",
    "sorghum",
    "sunflower",
    "other_cereals",
    "oilseeds_other",
    "legumes",
    "forage_hay",
    "fallow",
}

DROP_CLASSES = {
    "double_crop",
    "developed_water_wetlands",
    "non_ag_natural",
    "pasture_grass",
    "orchards_vineyards_tree_crops",
}

OPTIONAL_CLASSES = {
    "cotton",
    "rice",
    "peanuts",
    "root_tubers",
    "vegetables_melons",
    "fruits_berries",
    "sugar_crops",
    "specialty_crops",
}

## Этап 3. Функции подготовки и словарь названий
Собираем reusable-функции и справочник code -> crop name для создания *_name и *_group колонок.

### 3.1 Helper-функции подготовки
Выделяем функции поиска CDL-колонок, маппинга имен/групп и удаления технических колонок.

In [8]:
CDL_YEARS = list(range(2017, 2025))
DEFAULT_DROP_COLUMNS = ["Shape_Length", "Shape_Area", "CSBYEARS"]

def get_cdl_columns(df: pd.DataFrame, years=CDL_YEARS):
    expected_columns = [f"CDL{year}" for year in years]
    cdl_columns = [col for col in expected_columns if col in df.columns]
    missing_columns = [col for col in expected_columns if col not in df.columns]
    return cdl_columns, missing_columns

def add_cdl_name_columns(
    df: pd.DataFrame,
    cdl_columns,
    code_to_name: dict,
    unknown_label: str = "Unknown",
):
    result = df.copy()
    unknown_name_counts = {}

    for code_col in cdl_columns:
        mapped_names = result[code_col].map(code_to_name)
        name_col = f"{code_col}_name"
        result[name_col] = mapped_names.fillna(unknown_label)
        unknown_name_counts[name_col] = int(mapped_names.isna().sum())

    return result, unknown_name_counts

def add_cdl_group_columns(
    df: pd.DataFrame,
    cdl_columns,
    code_to_group: dict,
    unknown_label: str = "other_unknown",
):
    result = df.copy()
    unknown_group_counts = {}

    for code_col in cdl_columns:
        mapped_groups = result[code_col].map(code_to_group)
        group_col = f"{code_col}_group"
        result[group_col] = mapped_groups.fillna(unknown_label)
        unknown_group_counts[group_col] = int(mapped_groups.isna().sum())

    return result, unknown_group_counts

def drop_unneeded_columns(df: pd.DataFrame, columns_to_drop=DEFAULT_DROP_COLUMNS):
    existing_to_drop = [col for col in columns_to_drop if col in df.columns]
    result = df.drop(columns=existing_to_drop).copy()
    return result, existing_to_drop

### 3.1.1 Оркестрация подготовки
Объединяем helper-функции в единый пайплайн подготовки и отдельную печать диагностики.

In [9]:
def prepare_cdl_dataset(
    df: pd.DataFrame,
    code_to_name: dict,
    code_to_group: dict,
    years=CDL_YEARS,
    columns_to_drop=DEFAULT_DROP_COLUMNS,
    unknown_name_label: str = "Unknown",
    unknown_group_label: str = "other_unknown",
):
    result = df.copy()
    cdl_columns, missing_columns = get_cdl_columns(result, years=years)

    if not cdl_columns:
        raise ValueError("No CDL columns found for the specified years.")

    result, unknown_name_counts = add_cdl_name_columns(
        result,
        cdl_columns=cdl_columns,
        code_to_name=code_to_name,
        unknown_label=unknown_name_label,
    )
    result, unknown_group_counts = add_cdl_group_columns(
        result,
        cdl_columns=cdl_columns,
        code_to_group=code_to_group,
        unknown_label=unknown_group_label,
    )
    result, dropped_columns = drop_unneeded_columns(
        result,
        columns_to_drop=columns_to_drop,
    )

    diagnostics = {
        "cdl_columns": cdl_columns,
        "missing_cdl_columns": missing_columns,
        "added_name_columns": [f"{col}_name" for col in cdl_columns],
        "added_group_columns": [f"{col}_group" for col in cdl_columns],
        "dropped_columns": dropped_columns,
        "unknown_name_counts": unknown_name_counts,
        "unknown_group_counts": unknown_group_counts,
    }
    return result, diagnostics

def print_preparation_diagnostics(diagnostics: dict):
    print("CDL columns found:", diagnostics["cdl_columns"])
    print("Missing CDL columns:", diagnostics["missing_cdl_columns"])
    print("Dropped technical columns:", diagnostics["dropped_columns"])
    unknown_names_total = sum(diagnostics["unknown_name_counts"].values())
    unknown_groups_total = sum(diagnostics["unknown_group_counts"].values())
    print("Total unknown names:", unknown_names_total)
    print("Total unknown groups:", unknown_groups_total)

### 3.2 Справочник CDL code -> crop name
Используем отдельный словарь названий культур для колонок вида CDL{year}_name.

In [10]:
CDL_CODE_TO_NAME = {
    1: "Corn",
    2: "Cotton",
    3: "Rice",
    4: "Sorghum",
    5: "Soybeans",
    6: "Sunflower",

    10: "Peanuts",
    11: "Tobacco",
    12: "Sweet Corn",
    13: "Pop or Orn Corn",
    14: "Mint",

    21: "Barley",
    22: "Durum Wheat",
    23: "Spring Wheat",
    24: "Winter Wheat",
    25: "Other Small Grains",
    26: "Dbl Crop WinWht/Soybeans",
    27: "Rye",
    28: "Oats",
    29: "Millet",
    30: "Speltz",

    31: "Canola",
    32: "Flaxseed",
    33: "Safflower",
    34: "Rape Seed",
    35: "Mustard",
    36: "Alfalfa",
    37: "Other Hay/Non Alfalfa",
    38: "Camelina",
    39: "Buckwheat",

    41: "Sugarbeets",
    42: "Dry Beans",
    43: "Potatoes",
    44: "Other Crops",
    45: "Sugarcane",
    46: "Sweet Potatoes",
    47: "Misc Vegs & Fruits",
    48: "Watermelons",
    49: "Onions",
    50: "Cucumbers",
    51: "Chick Peas",
    52: "Lentils",
    53: "Peas",
    54: "Tomatoes",
    55: "Caneberries",
    56: "Hops",
    57: "Herbs",
    58: "Clover/Wildflowers",
    59: "Sod/Grass Seed",
    60: "Switchgrass",
    61: "Fallow/Idle Cropland",
    62: "Pasture/Grass",
    63: "Forest",
    64: "Shrubland",
    65: "Barren",
    66: "Cherries",
    67: "Peaches",
    68: "Apples",
    69: "Grapes",
    70: "Christmas Trees",
    71: "Other Tree Crops",
    72: "Citrus",
    74: "Pecans",
    75: "Almonds",
    76: "Walnuts",
    77: "Pears",

    81: "Clouds/No Data",
    82: "Developed",
    83: "Water",
    87: "Wetlands",
    88: "Nonag/Undefined",
    92: "Aquaculture",

    111: "Open Water",
    112: "Perennial Ice/Snow",
    121: "Developed/Open Space",
    122: "Developed/Low Intensity",
    123: "Developed/Med Intensity",
    124: "Developed/High Intensity",
    131: "Barren",
    141: "Deciduous Forest",
    142: "Evergreen Forest",
    143: "Mixed Forest",
    152: "Shrubland",
    176: "Grassland/Pasture",
    190: "Woody Wetlands",
    195: "Herbaceous Wetlands",

    204: "Pistachios",
    205: "Triticale",
    206: "Carrots",
    207: "Asparagus",
    208: "Garlic",
    209: "Cantaloupes",
    210: "Prunes",
    211: "Olives",
    212: "Oranges",
    213: "Honeydew Melons",
    214: "Broccoli",
    215: "Avocados",
    216: "Peppers",
    217: "Pomegranates",
    218: "Nectarines",
    219: "Greens",
    220: "Plums",
    221: "Strawberries",
    222: "Squash",
    223: "Apricots",
    224: "Vetch",
    225: "Dbl Crop WinWht/Corn",
    226: "Dbl Crop Oats/Corn",
    227: "Lettuce",
    228: "Dbl Crop Triticale/Corn",
    229: "Pumpkins",
    230: "Dbl Crop Lettuce/Durum Wht",
    231: "Dbl Crop Lettuce/Cantaloupe",
    232: "Dbl Crop Lettuce/Cotton",
    233: "Dbl Crop Lettuce/Barley",
    234: "Dbl Crop Durum Wht/Sorghum",
    235: "Dbl Crop Barley/Sorghum",
    236: "Dbl Crop WinWht/Sorghum",
    237: "Dbl Crop Barley/Corn",
    238: "Dbl Crop WinWht/Cotton",
    239: "Dbl Crop Soybeans/Cotton",
    240: "Dbl Crop Soybeans/Oats",
    241: "Dbl Crop Corn/Soybeans",
    242: "Blueberries",
    243: "Cabbage",
    244: "Cauliflower",
    245: "Celery",
    246: "Radishes",
    247: "Turnips",
    248: "Eggplants",
    249: "Gourds",
    250: "Cranberries",
    254: "Dbl Crop Barley/Soybeans",
}

## Этап 4. Применение подготовки датасета
Запускаем подготовку на рабочей копии, печатаем диагностику и проверяем результат.

In [11]:
df_prepared, prep_diagnostics = prepare_cdl_dataset(
    df=df,
    code_to_name=CDL_CODE_TO_NAME,
    code_to_group=CDL_AGGREGATION,
    years=CDL_YEARS,
    columns_to_drop=DEFAULT_DROP_COLUMNS,
    unknown_name_label="Unknown",
    unknown_group_label="other_unknown",
)

print_preparation_diagnostics(prep_diagnostics)
print("Original shape:", df.shape)
print("Prepared shape:", df_prepared.shape)

CDL columns found: ['CDL2017', 'CDL2018', 'CDL2019', 'CDL2020', 'CDL2021', 'CDL2022', 'CDL2023', 'CDL2024']
Missing CDL columns: []
Dropped technical columns: ['Shape_Length', 'Shape_Area', 'CSBYEARS']
Total unknown names: 115
Total unknown groups: 115
Original shape: (16418258, 20)
Prepared shape: (16418258, 33)


### 4.1 Проверка подготовленного датасета
Смотрим ключевые колонки, пример строк и убеждаемся, что исходный df не был изменен.

In [107]:
check_columns = ["CSBID", "CSBACRES", "STATEFIPS", "CNTYFIPS"]
first_cdl_cols = prep_diagnostics["cdl_columns"][:2]
for code_col in first_cdl_cols:
    check_columns.extend([code_col, f"{code_col}_name", f"{code_col}_group"])

check_columns = [col for col in check_columns if col in df_prepared.columns]
display(df_prepared[check_columns].head(5))

print(
    "Original technical columns still in df:",
    [col for col in DEFAULT_DROP_COLUMNS if col in df.columns],
)
print(
    "Technical columns in df_prepared:",
    [col for col in DEFAULT_DROP_COLUMNS if col in df_prepared.columns],
)

,CSBID,CSBACRES,STATEFIPS,CNTYFIPS,CDL2017,CDL2017_name,CDL2017_group,CDL2018,CDL2018_name,CDL2018_group
0,351724000000001,2.777993,35,059,24,Winter Wheat,wheat,24,Winter Wheat,wheat
1,351724000000002,9.293377,35,059,24,Winter Wheat,wheat,24,Winter Wheat,wheat
2,351724000000003,5.775892,35,059,24,Winter Wheat,wheat,24,Winter Wheat,wheat
3,351724000000004,5.206396,35,059,24,Winter Wheat,wheat,24,Winter Wheat,wheat
4,351724000000005,50.284333,35,059,24,Winter Wheat,wheat,24,Winter Wheat,wheat


Original technical columns still in df: ['Shape_Length', 'Shape_Area', 'CSBYEARS']
Technical columns in df_prepared: []


## Этап 5. Диагностика неизвестных кодов и очистка
Оцениваем unknown-коды и удаляем строки с нежелательными классами перед построением последовательностей.

In [12]:
def build_unknown_code_report(
    df_prepared: pd.DataFrame,
    cdl_columns,
    unknown_name_label: str = "Unknown",
    unknown_group_label: str = "other_unknown",
):
    rows = []

    for code_col in cdl_columns:
        name_col = f"{code_col}_name"
        group_col = f"{code_col}_group"
        if name_col not in df_prepared.columns or group_col not in df_prepared.columns:
            continue

        mask_unknown = (
            (df_prepared[name_col] == unknown_name_label)
            | (df_prepared[group_col] == unknown_group_label)
        )

        if not mask_unknown.any():
            continue

        counts = df_prepared.loc[mask_unknown, code_col].value_counts(dropna=False)
        year = int(code_col.replace("CDL", "")) if code_col.startswith("CDL") else None

        for code_value, count in counts.items():
            rows.append(
                {
                    "year": year,
                    "cdl_column": code_col,
                    "code": code_value,
                    "count": int(count),
                }
            )

    unknown_by_year = pd.DataFrame(rows)
    if not unknown_by_year.empty:
        unknown_by_year = unknown_by_year.sort_values(
            ["count", "year", "code"], ascending=[False, True, True]
        ).reset_index(drop=True)
        unknown_by_code = (
            unknown_by_year.groupby("code", dropna=False)["count"]
            .sum()
            .to_frame(name="count")
            .reset_index()
            .sort_values(by="count", ascending=False)
            .reset_index(drop=True)
        )
    else:
        unknown_by_year = pd.DataFrame(columns=["year", "cdl_column", "code", "count"])
        unknown_by_code = pd.DataFrame(columns=["code", "count"])

    return unknown_by_year, unknown_by_code

### 5.1 Unknown-коды после маппинга
Строим таблицы unknown-кодов по годам и суммарно по коду для контроля покрытия словарей.

In [13]:
unknown_by_year, unknown_by_code = build_unknown_code_report(
    df_prepared=df_prepared,
    cdl_columns=prep_diagnostics["cdl_columns"],
    unknown_name_label="Unknown",
    unknown_group_label="other_unknown",
)

unknown_codes = unknown_by_code["code"].dropna().tolist()

print("Unknown rows total:", int(unknown_by_year["count"].sum()))
print("Unique unknown codes:", len(unknown_codes))
print("Unknown code list:", unknown_codes)
display(unknown_by_code)
display(unknown_by_year.head(50))

Unknown rows total: 115
Unique unknown codes: 1
Unknown code list: [0]


,code,count
0,0,115


,year,cdl_column,code,count
0,2024,CDL2024,0,69
1,2022,CDL2022,0,9
2,2019,CDL2019,0,8
3,2021,CDL2021,0,7
4,2017,CDL2017,0,6
5,2023,CDL2023,0,6
6,2018,CDL2018,0,5
7,2020,CDL2020,0,5


### 5.2 Фильтрация строк с bad/unknown классами
Удаляем строки, где в любом году встретились нежелательные группы или Unknown.

In [14]:
# Remove rows where at least one year contains a drop class or unknown
group_columns = [
    col for col in prep_diagnostics.get("added_group_columns", [])
    if col in df_prepared.columns
]
name_columns = [
    col for col in prep_diagnostics.get("added_name_columns", [])
    if col in df_prepared.columns
]

bad_group_classes = set(DROP_CLASSES) | {"other_unknown"}
mask_bad_group = (
    df_prepared[group_columns].isin(bad_group_classes).any(axis=1)
    if group_columns
    else pd.Series(False, index=df_prepared.index, dtype=bool)
)
mask_unknown_name = (
    (df_prepared[name_columns] == "Unknown").any(axis=1)
    if name_columns
    else pd.Series(False, index=df_prepared.index, dtype=bool)
)
mask_drop = mask_bad_group | mask_unknown_name

rows_before_cleanup = len(df_prepared)
df_filtered = df_prepared.loc[~mask_drop].copy()
rows_after_cleanup = len(df_filtered)

print("Rows before filtering:", rows_before_cleanup)
print("Rows removed:", int(mask_drop.sum()))
print("Rows after filtering:", rows_after_cleanup)
print("Shape before:", df_prepared.shape)
print("Shape after:", df_filtered.shape)

Rows before filtering: 16418258
Rows removed: 8307388
Rows after filtering: 8110870
Shape before: (16418258, 33)
Shape after: (8110870, 33)


### 5.3 Диагностика удаленных строк
Смотрим распределение плохих групп в удаленных строках и проверяем сэмпл очищенного датасета.

In [15]:
if mask_drop.any() and group_columns:
    removed_bad_group_counts = (
        df_prepared.loc[mask_drop, group_columns]
        .stack()
        .value_counts()
        .rename_axis("group")
        .reset_index(name="count")
    )
    removed_bad_group_counts = removed_bad_group_counts[
        removed_bad_group_counts["group"].isin(bad_group_classes)
    ]
    print("Bad/unknown groups in removed rows:")
    display(removed_bad_group_counts)

display(df_filtered.head(5))

Bad/unknown groups in removed rows:


,group,count
1,pasture_grass,14990700
6,developed_water_wetlands,2247685
7,non_ag_natural,2140121
8,orchards_vineyards_tree_crops,2052559
9,double_crop,1596792
23,other_unknown,115


,CSBID,CSBACRES,CDL2017,CDL2018,CDL2019,CDL2020,CDL2021,CDL2022,CDL2023,CDL2024,...,CDL2023_name,CDL2024_name,CDL2017_group,CDL2018_group,CDL2019_group,CDL2020_group,CDL2021_group,CDL2022_group,CDL2023_group,CDL2024_group
35,351724000000036,20.942217,24,24,4,4,61,61,61,2,...,Fallow/Idle Cropland,Cotton,wheat,wheat,sorghum,sorghum,fallow,fallow,fallow,cotton
36,351724000000037,10.916051,24,24,4,4,61,61,4,2,...,Sorghum,Cotton,wheat,wheat,sorghum,sorghum,fallow,fallow,sorghum,cotton
43,351724000000044,17.243968,36,36,28,61,61,24,4,24,...,Sorghum,Winter Wheat,forage_hay,forage_hay,other_cereals,fallow,fallow,wheat,sorghum,wheat
44,351724000000045,5.108278,36,36,28,4,61,24,4,24,...,Sorghum,Winter Wheat,forage_hay,forage_hay,other_cereals,sorghum,fallow,wheat,sorghum,wheat
48,351724000000049,44.854487,36,36,28,4,61,24,4,24,...,Sorghum,Winter Wheat,forage_hay,forage_hay,other_cereals,sorghum,fallow,wheat,sorghum,wheat


## Этап 6. EDA по очищенным группам и сборка feature-table
Проверяем распределения классов, редкие группы и формируем df_model для окна последовательностей.

### 6.0 Базовые распределения по группам
Считаем частоты культур по всем годам и внутри каждого года.

In [16]:
# Final dataframe for analysis (no inplace changes to source data)
final_df = df_filtered.copy()

# Target and group columns
target_col = "CDL2024_group"
group_columns = [
    f"CDL{year}_group" for year in range(2017, 2025)
    if f"CDL{year}_group" in final_df.columns
]

if not group_columns:
    raise ValueError("No CDL*_group columns found in final_df")
if target_col not in final_df.columns:
    raise ValueError(f"Target column not found: {target_col}")

summary_all_years = (
    final_df[group_columns]
    .stack()
    .value_counts(dropna=False)
    .rename_axis("group")
    .reset_index(name="count")
)
summary_all_years["share"] = summary_all_years["count"] / summary_all_years["count"].sum()

per_year_long = final_df[group_columns].melt(var_name="year_col", value_name="group")
per_year_long["year"] = per_year_long["year_col"].str.extract(
    r"CDL(\d{4})_group", expand=False
)
per_year_long["year"] = pd.to_numeric(per_year_long["year"], errors="coerce")
per_year_long = per_year_long.dropna(subset=["year"]).copy()
per_year_long["year"] = per_year_long["year"].astype(int)

summary_per_year = (
    per_year_long.groupby(["year", "group"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["year", "count"], ascending=[True, False])
    .reset_index(drop=True)
)
summary_per_year["share_within_year"] = (
    summary_per_year["count"]
    / summary_per_year.groupby("year")["count"].transform("sum")
)

print("Final df shape:", final_df.shape)
print("Target column:", target_col)
print("Group columns:", group_columns)
print("\nsummary_all_years (head):")
display(summary_all_years.head(20))
print("summary_per_year (head):")
display(summary_per_year.head(30))

Final df shape: (8110870, 33)
Target column: CDL2024_group
Group columns: ['CDL2017_group', 'CDL2018_group', 'CDL2019_group', 'CDL2020_group', 'CDL2021_group', 'CDL2022_group', 'CDL2023_group', 'CDL2024_group']

summary_all_years (head):


,group,count,share
0,corn,18431742,0.284059
1,soybeans,17438163,0.268747
2,wheat,9156895,0.141121
3,forage_hay,6101579,0.094034
4,fallow,3361364,0.051803
5,cotton,3069392,0.047304
6,sorghum,1700538,0.026208
7,other_cereals,1541381,0.023755
8,legumes,955482,0.014725
9,oilseeds_other,600755,0.009258


summary_per_year (head):


,year,group,count,share_within_year
0,2017,soybeans,2311905,0.285038
1,2017,corn,2252019,0.277654
2,2017,wheat,1156836,0.142628
3,2017,forage_hay,792938,0.097762
4,2017,fallow,386436,0.047644
5,2017,cotton,363018,0.044757
6,2017,sorghum,177249,0.021853
7,2017,other_cereals,167521,0.020654
8,2017,legumes,144656,0.017835
9,2017,oilseeds_other,68277,0.008418


### 6.1 Нежелательные и редкие классы
Проверяем, остались ли нежелательные группы, и выделяем редкие классы по выбранному порогу.

In [17]:
undesired_classes = [
    "double_crop",
    "developed_water_wetlands",
    "non_ag_natural",
    "pasture_grass",
    "orchards_vineyards_tree_crops",
    "other_unknown",
]

undesired_left_all = summary_all_years[
    summary_all_years["group"].isin(undesired_classes)
] .copy()
undesired_left_per_year = summary_per_year[
    summary_per_year["group"].isin(undesired_classes)
] .copy()

target_summary = (
    final_df[target_col]
    .value_counts(dropna=False)
    .rename_axis("target")
    .reset_index(name="count")
)
target_summary["share"] = target_summary["count"] / target_summary["count"].sum()

RARE_SHARE_THRESHOLD = 0.01
rare_classes_all_years = summary_all_years[
    summary_all_years["share"] < RARE_SHARE_THRESHOLD
] .copy()
rare_target_classes = target_summary[
    target_summary["share"] < RARE_SHARE_THRESHOLD
] .copy()

print("\nUndesired classes left (all years):")
if undesired_left_all.empty:
    print("No undesired classes found.")
else:
    display(undesired_left_all)

print("\nUndesired classes left (per year):")
if undesired_left_per_year.empty:
    print("No undesired classes found.")
else:
    display(undesired_left_per_year)

print(f"\nRare classes threshold: share < {RARE_SHARE_THRESHOLD}")
print("Rare classes across all years:")
display(rare_classes_all_years)
print("Rare target classes:")
display(rare_target_classes)
print("target_summary:")
display(target_summary)


Undesired classes left (all years):
No undesired classes found.

Undesired classes left (per year):
No undesired classes found.

Rare classes threshold: share < 0.01
Rare classes across all years:


,group,count,share
9,oilseeds_other,600755,0.009258
10,sugar_crops,540044,0.008323
11,peanuts,503686,0.007763
12,rice,412495,0.006357
13,root_tubers,301440,0.004646
14,sunflower,273278,0.004212
15,vegetables_melons,234616,0.003616
16,specialty_crops,223133,0.003439
17,fruits_berries,40977,0.000632


Rare target classes:


,target,count,share
10,peanuts,74965,0.009243
11,sugar_crops,65006,0.008015
12,rice,59181,0.007297
13,root_tubers,42666,0.005260
14,specialty_crops,41884,0.005164
15,vegetables_melons,41690,0.005140
16,sunflower,23216,0.002862
17,fruits_berries,4984,0.000614


target_summary:


,target,count,share
0,corn,2334385,0.287809
1,soybeans,2242871,0.276527
2,wheat,1127385,0.138997
3,forage_hay,665157,0.082008
4,cotton,358550,0.044206
5,fallow,338558,0.041741
6,sorghum,229716,0.028322
7,other_cereals,217988,0.026876
8,legumes,148665,0.018329
9,oilseeds_other,94003,0.011590


### 6.2 Формирование признаков для модели
Собираем список признаков, фиксируем исключенные колонки и формируем рабочий df_model.

In [18]:
history_group_columns = [col for col in group_columns if col != target_col]
base_numeric_geo_columns = [
    col for col in ["CSBID", "CSBACRES", "STATEFIPS", "CNTYFIPS", "INSIDE_X", "INSIDE_Y"]
    if col in final_df.columns
]
base_feature_columns = base_numeric_geo_columns + history_group_columns
optional_feature_columns = [
    col for col in ["STATEASD", "ASD", "CNTY"] if col in final_df.columns
]

name_columns_to_exclude = [col for col in final_df.columns if col.endswith("_name")]
raw_cdl_code_columns_to_exclude = [
    col for col in final_df.columns
    if col.startswith("CDL") and col[3:].isdigit() and f"{col}_group" in final_df.columns
]

df_model_columns = base_feature_columns + optional_feature_columns + [target_col]
df_model = final_df[df_model_columns].copy()

print("df_model shape:", df_model.shape)
print("\nBase feature columns:", base_feature_columns)
print("Optional feature columns:", optional_feature_columns)
print("Excluded *_name columns:", name_columns_to_exclude)
print("Excluded raw CDL code columns:", raw_cdl_code_columns_to_exclude)
print("df_model sample:")
display(df_model.head(5))

df_model shape: (8110870, 17)

Base feature columns: ['CSBID', 'CSBACRES', 'STATEFIPS', 'CNTYFIPS', 'INSIDE_X', 'INSIDE_Y', 'CDL2017_group', 'CDL2018_group', 'CDL2019_group', 'CDL2020_group', 'CDL2021_group', 'CDL2022_group', 'CDL2023_group']
Optional feature columns: ['STATEASD', 'ASD', 'CNTY']
Excluded *_name columns: ['CDL2017_name', 'CDL2018_name', 'CDL2019_name', 'CDL2020_name', 'CDL2021_name', 'CDL2022_name', 'CDL2023_name', 'CDL2024_name']
Excluded raw CDL code columns: ['CDL2017', 'CDL2018', 'CDL2019', 'CDL2020', 'CDL2021', 'CDL2022', 'CDL2023', 'CDL2024']
df_model sample:


,CSBID,CSBACRES,STATEFIPS,CNTYFIPS,INSIDE_X,INSIDE_Y,CDL2017_group,CDL2018_group,CDL2019_group,CDL2020_group,CDL2021_group,CDL2022_group,CDL2023_group,STATEASD,ASD,CNTY,CDL2024_group
35,351724000000036,20.942217,35,021,-664007.6193,1.445271e+06,wheat,wheat,sorghum,sorghum,fallow,fallow,fallow,3530,30,Harding,cotton
36,351724000000037,10.916051,35,021,-664304.1197,1.445202e+06,wheat,wheat,sorghum,sorghum,fallow,fallow,sorghum,3530,30,Harding,cotton
43,351724000000044,17.243968,35,021,-664688.1027,1.445128e+06,forage_hay,forage_hay,other_cereals,fallow,fallow,wheat,sorghum,3530,30,Harding,wheat
44,351724000000045,5.108278,35,021,-664794.7219,1.445021e+06,forage_hay,forage_hay,other_cereals,sorghum,fallow,wheat,sorghum,3530,30,Harding,wheat
48,351724000000049,44.854487,35,021,-665083.2265,1.445121e+06,forage_hay,forage_hay,other_cereals,sorghum,fallow,wheat,sorghum,3530,30,Harding,wheat


## Этап 7. Построение окон history -> target
Строим sliding-window датасет, анализируем target и подготавливаем window_df_filtered для моделирования.

In [19]:
def build_sliding_window_dataset(
    df: pd.DataFrame,
    years=range(2017, 2025),
    window_size: int = 3,
    id_col: str = "CSBID",
    group_suffix: str = "_group",
    extra_feature_columns=None,
) -> pd.DataFrame:
    years = list(years)
    if window_size <= 0:
        raise ValueError("window_size must be > 0")
    if len(years) <= window_size:
        raise ValueError("Need more years than window_size to build targets")

    if extra_feature_columns is None:
        extra_feature_columns = ["CSBACRES", "STATEFIPS", "CNTYFIPS", "INSIDE_X", "INSIDE_Y"]

    group_columns = [f"CDL{year}{group_suffix}" for year in years]
    missing_group_columns = [col for col in group_columns if col not in df.columns]
    if missing_group_columns:
        raise ValueError(f"Missing group columns: {missing_group_columns}")

    fixed_columns = [id_col] + [col for col in extra_feature_columns if col in df.columns]
    frames = []

    for start_idx in range(len(years) - window_size):
        history_years = years[start_idx : start_idx + window_size]
        target_year = years[start_idx + window_size]

        history_cols = [f"CDL{year}{group_suffix}" for year in history_years]
        target_col = f"CDL{target_year}{group_suffix}"
        select_cols = [col for col in fixed_columns + history_cols + [target_col] if col in df.columns]

        window_part = df[select_cols].copy()
        rename_map = {
            history_cols[0]: "history_1",
            history_cols[1]: "history_2",
            history_cols[2]: "history_3",
            target_col: "target",
        }
        window_part = window_part.rename(columns=rename_map)
        window_part["target_year"] = target_year

        ordered_cols = [
            id_col,
            "target_year",
            "history_1",
            "history_2",
            "history_3",
            "target",
            "CSBACRES",
            "STATEFIPS",
            "CNTYFIPS",
            "INSIDE_X",
            "INSIDE_Y",
        ]
        window_part = window_part[[col for col in ordered_cols if col in window_part.columns]]
        frames.append(window_part)

    if not frames:
        return pd.DataFrame(columns=[
            id_col,
            "target_year",
            "history_1",
            "history_2",
            "history_3",
            "target",
            "CSBACRES",
            "STATEFIPS",
            "CNTYFIPS",
            "INSIDE_X",
            "INSIDE_Y",
        ])

    return pd.concat(frames, ignore_index=True)

### 7.1 Сборка window_df
Берем очищенный источник и разворачиваем его в пары history_1..history_3 -> target.

In [20]:
df_final_candidate = globals().get("df_final")
if df_final_candidate is not None:
    df_source = df_final_candidate.copy()
elif "final_df" in globals():
    df_source = final_df.copy()
else:
    raise ValueError("Expected df_final or final_df in notebook variables")

window_df = build_sliding_window_dataset(
    df=df_source,
    years=range(2017, 2025),
    window_size=3,
    id_col="CSBID",
    group_suffix="_group",
    extra_feature_columns=["CSBACRES", "STATEFIPS", "CNTYFIPS", "INSIDE_X", "INSIDE_Y"],
)

print("Source dataframe shape:", df_source.shape)
print("window_df shape:", window_df.shape)
print("Columns in window_df:", window_df.columns.tolist())
display(window_df.head(10))

Source dataframe shape: (8110870, 33)
window_df shape: (40554350, 11)
Columns in window_df: ['CSBID', 'target_year', 'history_1', 'history_2', 'history_3', 'target', 'CSBACRES', 'STATEFIPS', 'CNTYFIPS', 'INSIDE_X', 'INSIDE_Y']


,CSBID,target_year,history_1,history_2,history_3,target,CSBACRES,STATEFIPS,CNTYFIPS,INSIDE_X,INSIDE_Y
0,351724000000036,2020,wheat,wheat,sorghum,sorghum,20.942217,35,021,-664007.6193,1.445271e+06
1,351724000000037,2020,wheat,wheat,sorghum,sorghum,10.916051,35,021,-664304.1197,1.445202e+06
2,351724000000044,2020,forage_hay,forage_hay,other_cereals,fallow,17.243968,35,021,-664688.1027,1.445128e+06
3,351724000000045,2020,forage_hay,forage_hay,other_cereals,sorghum,5.108278,35,021,-664794.7219,1.445021e+06
4,351724000000049,2020,forage_hay,forage_hay,other_cereals,sorghum,44.854487,35,021,-665083.2265,1.445121e+06
5,351724000000052,2020,forage_hay,forage_hay,other_cereals,fallow,19.383838,35,021,-664608.1837,1.444829e+06
6,351724000000053,2020,forage_hay,forage_hay,other_cereals,sorghum,17.094740,35,021,-665081.8334,1.444775e+06
7,351724000000056,2020,forage_hay,forage_hay,wheat,wheat,2.659192,35,021,-665574.4173,1.444784e+06
8,351724000000057,2020,forage_hay,forage_hay,other_cereals,fallow,13.225904,35,021,-664806.6663,1.444772e+06
9,351724000000058,2020,forage_hay,forage_hay,wheat,sorghum,46.355434,35,021,-665436.3124,1.444853e+06


### 7.2 Распределение target в window_df
Считаем частоты target и выделяем редкие классы для последующей фильтрации.

In [21]:
# Target analysis for window dataset
target_summary = (
    window_df["target"]
    .value_counts(dropna=False)
    .rename_axis("target")
    .reset_index(name="count")
)
target_summary["share"] = target_summary["count"] / target_summary["count"].sum()

rare_target_classes = target_summary.loc[
    target_summary["share"] < 0.01, "target"
].tolist()

print("window_df rows:", len(window_df))
print("Unique target classes:", target_summary.shape[0])
print("Rare target classes (share < 0.01):", rare_target_classes)
display(target_summary)

window_df rows: 40554350
Unique target classes: 18
Rare target classes (share < 0.01): ['oilseeds_other', 'sugar_crops', 'peanuts', 'rice', 'root_tubers', 'sunflower', 'specialty_crops', 'vegetables_melons', 'fruits_berries']


,target,count,share
0,corn,11752817,0.289804
1,soybeans,10970284,0.270508
2,wheat,5648151,0.139274
3,forage_hay,3740641,0.092238
4,cotton,1849867,0.045615
5,fallow,1802576,0.044448
6,sorghum,1158288,0.028561
7,other_cereals,1008139,0.024859
8,legumes,580447,0.014313
9,oilseeds_other,398305,0.009822


### 7.3 Фильтрация редких target-классов
Удаляем окна с редкими целевыми классами и фиксируем обновленное распределение.

In [22]:
# Filter out windows with rare target classes
rows_before_filter = len(window_df)
window_df_filtered = window_df.loc[
    ~window_df["target"].isin(rare_target_classes)
].copy()
rows_after_filter = len(window_df_filtered)

removed_target_classes = sorted(set(rare_target_classes))

target_summary_filtered = (
    window_df_filtered["target"]
    .value_counts(dropna=False)
    .rename_axis("target")
    .reset_index(name="count")
)
target_summary_filtered["share"] = (
    target_summary_filtered["count"] / target_summary_filtered["count"].sum()
)

base_feature_columns = [
    "history_1",
    "history_2",
    "history_3",
    "CSBACRES",
    "STATEFIPS",
    "CNTYFIPS",
    "INSIDE_X",
    "INSIDE_Y",
]
base_feature_columns = [
    col for col in base_feature_columns if col in window_df_filtered.columns
]

print("Rows before rare-target filtering:", rows_before_filter)
print("Rows after rare-target filtering:", rows_after_filter)
print("Removed rows:", rows_before_filter - rows_after_filter)
print("Removed target classes:", removed_target_classes)
print("base_feature_columns:", base_feature_columns)
print("Updated target distribution after filtering:")
display(target_summary_filtered)
display(window_df_filtered.head(10))

Rows before rare-target filtering: 40554350
Rows after rare-target filtering: 38511210
Removed rows: 2043140
Removed target classes: ['fruits_berries', 'oilseeds_other', 'peanuts', 'rice', 'root_tubers', 'specialty_crops', 'sugar_crops', 'sunflower', 'vegetables_melons']
base_feature_columns: ['history_1', 'history_2', 'history_3', 'CSBACRES', 'STATEFIPS', 'CNTYFIPS', 'INSIDE_X', 'INSIDE_Y']
Updated target distribution after filtering:


,target,count,share
0,corn,11752817,0.305179
1,soybeans,10970284,0.284859
2,wheat,5648151,0.146663
3,forage_hay,3740641,0.097131
4,cotton,1849867,0.048035
5,fallow,1802576,0.046807
6,sorghum,1158288,0.030077
7,other_cereals,1008139,0.026178
8,legumes,580447,0.015072


,CSBID,target_year,history_1,history_2,history_3,target,CSBACRES,STATEFIPS,CNTYFIPS,INSIDE_X,INSIDE_Y
0,351724000000036,2020,wheat,wheat,sorghum,sorghum,20.942217,35,021,-664007.6193,1.445271e+06
1,351724000000037,2020,wheat,wheat,sorghum,sorghum,10.916051,35,021,-664304.1197,1.445202e+06
2,351724000000044,2020,forage_hay,forage_hay,other_cereals,fallow,17.243968,35,021,-664688.1027,1.445128e+06
3,351724000000045,2020,forage_hay,forage_hay,other_cereals,sorghum,5.108278,35,021,-664794.7219,1.445021e+06
4,351724000000049,2020,forage_hay,forage_hay,other_cereals,sorghum,44.854487,35,021,-665083.2265,1.445121e+06
5,351724000000052,2020,forage_hay,forage_hay,other_cereals,fallow,19.383838,35,021,-664608.1837,1.444829e+06
6,351724000000053,2020,forage_hay,forage_hay,other_cereals,sorghum,17.094740,35,021,-665081.8334,1.444775e+06
7,351724000000056,2020,forage_hay,forage_hay,wheat,wheat,2.659192,35,021,-665574.4173,1.444784e+06
8,351724000000057,2020,forage_hay,forage_hay,other_cereals,fallow,13.225904,35,021,-664806.6663,1.444772e+06
9,351724000000058,2020,forage_hay,forage_hay,wheat,sorghum,46.355434,35,021,-665436.3124,1.444853e+06


### 7.4 Проверка по отдельному полю
Контрольный просмотр окна для одного CSBID до и после фильтрации.

In [23]:
# Example: how many rows correspond to one specific CSBID
csbid_counts = (
    window_df_filtered["CSBID"]
    .value_counts()
    .rename_axis("CSBID")
    .reset_index(name="n_rows_filtered")
)
display(csbid_counts.head(10))

# You can set any CSBID manually, for example:
# id_to_check = 351724000000036
id_to_check = csbid_counts.iloc[0]["CSBID"]

rows_with_id_all = window_df.loc[window_df["CSBID"] == id_to_check].copy()
rows_with_id_filtered = window_df_filtered.loc[
    window_df_filtered["CSBID"] == id_to_check
].copy()

print("CSBID:", id_to_check)
print("Rows in window_df (before rare-target filtering):", len(rows_with_id_all))
print("Rows in window_df_filtered:", len(rows_with_id_filtered))
print(
    "Target years before filtering:",
    sorted(rows_with_id_all["target_year"].unique().tolist()),
)
print(
    "Target years after filtering:",
    sorted(rows_with_id_filtered["target_year"].unique().tolist()),
)

print("Rows for selected CSBID in filtered dataset:")
display(rows_with_id_filtered)

,CSBID,n_rows_filtered
0,351724000000036,5
1,351724000000037,5
2,351724000000044,5
3,351724000000045,5
4,351724000000049,5
5,351724000000052,5
6,351724000000053,5
7,351724000000056,5
8,351724000000057,5
9,351724000000058,5


CSBID: 351724000000036
Rows in window_df (before rare-target filtering): 5
Rows in window_df_filtered: 5
Target years before filtering: [2020, 2021, 2022, 2023, 2024]
Target years after filtering: [2020, 2021, 2022, 2023, 2024]
Rows for selected CSBID in filtered dataset:


,CSBID,target_year,history_1,history_2,history_3,target,CSBACRES,STATEFIPS,CNTYFIPS,INSIDE_X,INSIDE_Y
0,351724000000036,2020,wheat,wheat,sorghum,sorghum,20.942217,35,021,-664007.6193,1.445271e+06
8110870,351724000000036,2021,wheat,sorghum,sorghum,fallow,20.942217,35,021,-664007.6193,1.445271e+06
16221740,351724000000036,2022,sorghum,sorghum,fallow,fallow,20.942217,35,021,-664007.6193,1.445271e+06
24332610,351724000000036,2023,sorghum,fallow,fallow,fallow,20.942217,35,021,-664007.6193,1.445271e+06
32443480,351724000000036,2024,fallow,fallow,fallow,cotton,20.942217,35,021,-664007.6193,1.445271e+06


## Этап 8. Подготовка данных для моделирования
В этом разделе готовим таблицу признаков, кодируем target, делаем честный split по полям и запускаем baseline-модели для сравнения.

In [24]:
# Pre-baseline sanity checks (moved up from draft cells below)
id_column = "CSBID"
target_column = "target"
categorical_features = ["history_1", "history_2", "history_3", "STATEFIPS", "CNTYFIPS"]
numeric_features = ["CSBACRES", "INSIDE_X", "INSIDE_Y"]
base_feature_columns = categorical_features + numeric_features

print("Sanity check: target share sum")
if "target_summary_filtered" in globals():
    share_sum = target_summary_filtered["share"].sum()
    print("target_summary_filtered['share'].sum() =", float(share_sum))
elif "target_summary" in globals():
    share_sum = target_summary["share"].sum()
    print("target_summary['share'].sum() =", float(share_sum))
else:
    print("target summary table is not available yet")

check_cols = [id_column] + base_feature_columns + [target_column]
check_cols = [col for col in check_cols if col in window_df_filtered.columns]

print("\nDtypes for modeling columns:")
dtype_table = pd.DataFrame({
    "column": check_cols,
    "dtype": [str(window_df_filtered[col].dtype) for col in check_cols],
})
display(dtype_table)

print("\nMissing values in modeling columns:")
missing_table = (
    window_df_filtered[check_cols]
    .isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)
display(missing_table)

non_zero_missing = missing_table[missing_table["missing_count"] > 0]
if non_zero_missing.empty:
    print("No missing values in selected columns.")
else:
    print("Columns with missing values:")
    display(non_zero_missing)

Sanity check: target share sum
target_summary_filtered['share'].sum() = 1.0

Dtypes for modeling columns:


,column,dtype
0,CSBID,str
1,history_1,str
2,history_2,str
3,history_3,str
4,STATEFIPS,str
5,CNTYFIPS,str
6,CSBACRES,float64
7,INSIDE_X,float64
8,INSIDE_Y,float64
9,target,str



Missing values in modeling columns:


,column,missing_count
0,CSBID,0
1,history_1,0
2,history_2,0
3,history_3,0
4,STATEFIPS,0
5,CNTYFIPS,0
6,CSBACRES,0
7,INSIDE_X,0
8,INSIDE_Y,0
9,target,0


No missing values in selected columns.


In [25]:
# Baseline constants
RANDOM_STATE = 42

### 8.1 Проверки перед обучением
Проверяем типы и пропуски в ключевых колонках, чтобы не получить ошибки на этапе обучения и baseline-оценки.

### 8.2 Подготовка df_model
Разбиваем подготовку на несколько небольших шагов: проверка колонок, приведение типов, кодирование target, формирование списков признаков и финальная очистка строк.

In [26]:
# 8.2.1 Working copy and required columns
df_model = window_df_filtered.copy()

required_columns = [
    "CSBID",
    "target_year",
    "history_1",
    "history_2",
    "history_3",
    "target",
    "CSBACRES",
    "STATEFIPS",
    "CNTYFIPS",
    "INSIDE_X",
    "INSIDE_Y",
]
missing_required = [col for col in required_columns if col not in df_model.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

print("Initial df_model shape:", df_model.shape)
display(df_model.head(5))

Initial df_model shape: (38511210, 11)


,CSBID,target_year,history_1,history_2,history_3,target,CSBACRES,STATEFIPS,CNTYFIPS,INSIDE_X,INSIDE_Y
0,351724000000036,2020,wheat,wheat,sorghum,sorghum,20.942217,35,021,-664007.6193,1.445271e+06
1,351724000000037,2020,wheat,wheat,sorghum,sorghum,10.916051,35,021,-664304.1197,1.445202e+06
2,351724000000044,2020,forage_hay,forage_hay,other_cereals,fallow,17.243968,35,021,-664688.1027,1.445128e+06
3,351724000000045,2020,forage_hay,forage_hay,other_cereals,sorghum,5.108278,35,021,-664794.7219,1.445021e+06
4,351724000000049,2020,forage_hay,forage_hay,other_cereals,sorghum,44.854487,35,021,-665083.2265,1.445121e+06


In [27]:
# 8.2.2 Feature dtypes for CatBoost
categorical_history_cols = ["history_1", "history_2", "history_3"]
for col in categorical_history_cols:
    df_model[col] = df_model[col].astype("string")

df_model["STATEFIPS"] = df_model["STATEFIPS"].astype("string")
df_model["CNTYFIPS"] = df_model["CNTYFIPS"].astype("string")

numeric_features = ["CSBACRES", "INSIDE_X", "INSIDE_Y"]
for col in numeric_features:
    df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

df_model["target"] = df_model["target"].astype("string")

print("Feature dtype conversion completed.")
display(
    df_model[["history_1", "history_2", "history_3", "STATEFIPS", "CNTYFIPS", "target"]].head(5)
)

Feature dtype conversion completed.


,history_1,history_2,history_3,STATEFIPS,CNTYFIPS,target
0,wheat,wheat,sorghum,35,021,sorghum
1,wheat,wheat,sorghum,35,021,sorghum
2,forage_hay,forage_hay,other_cereals,35,021,fallow
3,forage_hay,forage_hay,other_cereals,35,021,sorghum
4,forage_hay,forage_hay,other_cereals,35,021,sorghum


In [28]:
# 8.2.3 Target mapping and encoded label
classes_in_df = set(df_model["target"].dropna().unique().tolist())
if "CDL_AGGREGATION" in globals():
    classes_in_order = [g for g in sorted(set(CDL_AGGREGATION.values())) if g in classes_in_df]
else:
    classes_in_order = sorted(classes_in_df)

target_to_id = {target_name: idx for idx, target_name in enumerate(classes_in_order)}
id_to_target = {idx: target_name for target_name, idx in target_to_id.items()}

df_model["target_encoded"] = df_model["target"].map(target_to_id)

mapping_df = (
    pd.DataFrame(
        {
            "target": list(target_to_id.keys()),
            "target_encoded": list(target_to_id.values()),
        }
    )
    .sort_values("target_encoded")
    .reset_index(drop=True)
)
print("Target mapping (target -> target_encoded):")
display(mapping_df)

Target mapping (target -> target_encoded):


,target,target_encoded
0,corn,0
1,cotton,1
2,fallow,2
3,forage_hay,3
4,legumes,4
5,other_cereals,5
6,sorghum,6
7,soybeans,7
8,wheat,8


In [29]:
# 8.2.4 Feature lists, missing report, and cleanup
categorical_features = ["history_1", "history_2", "history_3", "STATEFIPS", "CNTYFIPS"]
numeric_features = ["CSBACRES", "INSIDE_X", "INSIDE_Y"]
feature_columns = categorical_features + numeric_features

print("categorical_features:", categorical_features)
print("numeric_features:", numeric_features)
print("feature_columns:", feature_columns)

missing_report = (
    df_model[feature_columns + ["target_encoded"]]
    .isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)
display(missing_report)

missing_columns = missing_report.loc[missing_report["missing_count"] > 0, "column"].tolist()
if missing_columns:
    print("Columns with missing values:", missing_columns)
else:
    print("No missing values in feature_columns and target_encoded.")

rows_before_cleanup = len(df_model)
df_model = df_model.dropna(subset=["CSBID", "target_encoded"] + numeric_features).copy()
df_model["target_encoded"] = df_model["target_encoded"].astype(int)
rows_after_cleanup = len(df_model)

print("Rows before cleanup:", rows_before_cleanup)
print("Rows after cleanup:", rows_after_cleanup)
print("Rows removed during cleanup:", rows_before_cleanup - rows_after_cleanup)

categorical_features: ['history_1', 'history_2', 'history_3', 'STATEFIPS', 'CNTYFIPS']
numeric_features: ['CSBACRES', 'INSIDE_X', 'INSIDE_Y']
feature_columns: ['history_1', 'history_2', 'history_3', 'STATEFIPS', 'CNTYFIPS', 'CSBACRES', 'INSIDE_X', 'INSIDE_Y']


,column,missing_count
0,history_1,0
1,history_2,0
2,history_3,0
3,STATEFIPS,0
4,CNTYFIPS,0
5,CSBACRES,0
6,INSIDE_X,0
7,INSIDE_Y,0
8,target_encoded,0


No missing values in feature_columns and target_encoded.
Rows before cleanup: 38511210
Rows after cleanup: 38511210
Rows removed during cleanup: 0


### 8.3 Honest split и матрицы признаков
Сначала делим по уникальным полям CSBID без пересечений, затем в отдельном шаге строим X и y для train, validation и test.

In [30]:
# 8.3.1 Honest split by CSBID (70 / 15 / 15)
import numpy as np

unique_csbid = df_model["CSBID"].dropna().unique()
rng = np.random.default_rng(RANDOM_STATE)
shuffled_ids = rng.permutation(unique_csbid)

n_total = len(shuffled_ids)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)

train_ids = shuffled_ids[:n_train]
val_ids = shuffled_ids[n_train : n_train + n_val]
test_ids = shuffled_ids[n_train + n_val :]

train_df = df_model[df_model["CSBID"].isin(train_ids)].copy()
val_df = df_model[df_model["CSBID"].isin(val_ids)].copy()
test_df = df_model[df_model["CSBID"].isin(test_ids)].copy()

print("Rows in splits:")
print("train_df:", train_df.shape)
print("val_df:", val_df.shape)
print("test_df:", test_df.shape)

print("Unique CSBID in splits:")
print("train:", train_df["CSBID"].nunique())
print("val:", val_df["CSBID"].nunique())
print("test:", test_df["CSBID"].nunique())

train_set = set(train_ids.tolist())
val_set = set(val_ids.tolist())
test_set = set(test_ids.tolist())

train_val_overlap = train_set.intersection(val_set)
train_test_overlap = train_set.intersection(test_set)
val_test_overlap = val_set.intersection(test_set)

print("Overlap counts by CSBID:")
print("train ∩ val:", len(train_val_overlap))
print("train ∩ test:", len(train_test_overlap))
print("val ∩ test:", len(val_test_overlap))

if len(train_val_overlap) == 0 and len(train_test_overlap) == 0 and len(val_test_overlap) == 0:
    print("CSBID split check passed: no intersections.")
else:
    print("Warning: intersections found in CSBID split!")

Rows in splits:
train_df: (26958553, 12)
val_df: (5776530, 12)
test_df: (5776127, 12)
Unique CSBID in splits:
train: 5658862
val: 1212613
test: 1212614
Overlap counts by CSBID:
train ∩ val: 0
train ∩ test: 0
val ∩ test: 0
CSBID split check passed: no intersections.


In [31]:
# 8.3.2 Matrices and targets
X_train = train_df[feature_columns].copy()
X_val = val_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

y_train = train_df["target_encoded"].copy()
y_val = val_df["target_encoded"].copy()
y_test = test_df["target_encoded"].copy()

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val:", X_val.shape, "| y_val:", y_val.shape)
print("X_test:", X_test.shape, "| y_test:", y_test.shape)

X_train: (26958553, 8) | y_train: (26958553,)
X_val: (5776530, 8) | y_val: (5776530,)
X_test: (5776127, 8) | y_test: (5776127,)


### 8.4 Эвристические baseline
Сравниваем три простых подхода: Most Frequent, Last Crop и Markov 1-order. Сначала формируем предсказания по каждому baseline, затем считаем метрики и анализируем результаты.

In [128]:
# 8.4.1 Helper for baseline evaluation
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def evaluate_baseline(y_true, y_pred, model_name, split_name, id_to_target):
    id_to_target_norm = {int(k): v for k, v in id_to_target.items()}
    labels_sorted = sorted(id_to_target_norm.keys())
    target_names = [id_to_target_norm[label] for label in labels_sorted]

    y_true = pd.Series(y_true).astype(int)
    y_pred = pd.Series(y_pred, index=y_true.index).astype(int)

    metrics_row = {
        "model": model_name,
        "split": split_name,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
    }

    report_text = classification_report(
        y_true,
        y_pred,
        labels=labels_sorted,
        target_names=target_names,
        zero_division=0,
    )

    cm = confusion_matrix(y_true, y_pred, labels=labels_sorted)
    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{id_to_target_norm[label]}" for label in labels_sorted],
        columns=[f"pred_{id_to_target_norm[label]}" for label in labels_sorted],
    )

    return metrics_row, report_text, cm_df

In [129]:
# 8.4.2 Baseline Most Frequent Class
train_target_counts = train_df["target_encoded"].value_counts(dropna=False)
most_frequent_class_id = int(train_target_counts.index[0])
most_frequent_class_name = id_to_target.get(most_frequent_class_id, str(most_frequent_class_id))

y_val_pred_most_frequent = pd.Series(
    most_frequent_class_id,
    index=val_df.index,
    name="y_val_pred_most_frequent",
    dtype="int64",
)
y_test_pred_most_frequent = pd.Series(
    most_frequent_class_id,
    index=test_df.index,
    name="y_test_pred_most_frequent",
    dtype="int64",
)

print("Most frequent class from train_df:")
print("class_id:", most_frequent_class_id)
print("class_name:", most_frequent_class_name)
display(
    train_target_counts
    .rename_axis("target_encoded")
    .reset_index(name="count")
    .head(10)
)
print("Validation predictions shape:", y_val_pred_most_frequent.shape)
print("Test predictions shape:", y_test_pred_most_frequent.shape)

Most frequent class from train_df:
class_id: 0
class_name: corn


,target_encoded,count
0,0,8228378
1,7,7685795
2,8,3949428
3,3,2620758
4,1,1291090
5,2,1261227
6,6,809522
7,5,706181
8,4,406174


Validation predictions shape: (5776530,)
Test predictions shape: (5776127,)


In [130]:
# 8.4.3 Baseline Last Crop
def map_last_crop_predictions(split_df, split_name, target_to_id, fallback_class_id, id_to_target):
    pred_encoded = split_df["history_3"].map(target_to_id)
    unmapped_mask = pred_encoded.isna()

    if unmapped_mask.any():
        unmapped_values = (
            split_df.loc[unmapped_mask, "history_3"]
            .astype("string")
            .dropna()
            .unique()
            .tolist()
        )
        unmapped_values = sorted(unmapped_values)
        print(
            f"WARNING: {split_name} has {int(unmapped_mask.sum())} rows "
            "where history_3 is not mapped to target classes."
        )
        print(f"Unmapped values in {split_name} (up to 20):", unmapped_values[:20])
        if len(unmapped_values) > 20:
            print(f"Total unique unmapped values in {split_name}: {len(unmapped_values)}")

        pred_encoded = pred_encoded.fillna(fallback_class_id)
        fallback_name = id_to_target.get(int(fallback_class_id), str(int(fallback_class_id)))
        print(
            f"Fallback applied for {split_name}: class_id={int(fallback_class_id)}, "
            f"class_name={fallback_name}"
        )
    else:
        print(f"{split_name}: all history_3 values were mapped successfully.")

    return pred_encoded.astype(int), unmapped_mask

fallback_class_id = int(train_df["target_encoded"].value_counts().idxmax())

y_val_pred_last_crop, val_unmapped_mask_last_crop = map_last_crop_predictions(
    split_df=val_df,
    split_name="validation",
    target_to_id=target_to_id,
    fallback_class_id=fallback_class_id,
    id_to_target=id_to_target,
)
y_test_pred_last_crop, test_unmapped_mask_last_crop = map_last_crop_predictions(
    split_df=test_df,
    split_name="test",
    target_to_id=target_to_id,
    fallback_class_id=fallback_class_id,
    id_to_target=id_to_target,
)

val_last_crop_match_share = float((val_df["history_3"] == val_df["target"]).mean())
test_last_crop_match_share = float((test_df["history_3"] == test_df["target"]).mean())

print("Share(history_3 == target):")
print("validation:", f"{val_last_crop_match_share:.6f}")
print("test:", f"{test_last_crop_match_share:.6f}")

display(
    pd.DataFrame(
        {
            "split": ["validation", "test"],
            "rows": [len(val_df), len(test_df)],
            "unmapped_rows": [
                int(val_unmapped_mask_last_crop.sum()),
                int(test_unmapped_mask_last_crop.sum()),
            ],
            "share_history_3_eq_target": [
                val_last_crop_match_share,
                test_last_crop_match_share,
            ],
        }
    )
)

Unmapped values in validation (up to 20): ['fruits_berries', 'oilseeds_other', 'peanuts', 'rice', 'root_tubers', 'specialty_crops', 'sugar_crops', 'sunflower', 'vegetables_melons']
Fallback applied for validation: class_id=0, class_name=corn
Unmapped values in test (up to 20): ['fruits_berries', 'oilseeds_other', 'peanuts', 'rice', 'root_tubers', 'specialty_crops', 'sugar_crops', 'sunflower', 'vegetables_melons']
Fallback applied for test: class_id=0, class_name=corn
Share(history_3 == target):
validation: 0.354136
test: 0.355065


,split,rows,unmapped_rows,share_history_3_eq_target
0,validation,5776530,217867,0.354136
1,test,5776127,218586,0.355065


### 8.4.4 Baseline Markov 1-order
Оцениваем переходы P(target | history_1) только на train, затем делаем предсказания на validation/test с fallback на most frequent target из train.

In [141]:
# 8.4.4 Baseline Markov 1-order (history_1 -> target)
markov_1_transition_df = (
    train_df.groupby(["history_1", "target"])
    .size()
    .rename("count")
    .reset_index()
)

markov_1_transition_df["probability"] = (
    markov_1_transition_df["count"]
    / markov_1_transition_df.groupby("history_1")["count"].transform("sum")
)

markov_1_transition_df = markov_1_transition_df.sort_values(
    ["history_1", "count", "target"],
    ascending=[True, False, True],
).reset_index(drop=True)

markov_1_transition_df["target_encoded"] = markov_1_transition_df["target"].map(target_to_id)

markov_1_best_target_by_history_1 = (
    markov_1_transition_df
    .drop_duplicates(subset=["history_1"], keep="first")
    .set_index("history_1")["target"]
)

markov_1_fallback_target = str(train_df["target"].value_counts().idxmax())
markov_1_fallback_class_id = int(target_to_id[markov_1_fallback_target])


def predict_markov_1(split_df):
    pred_target = split_df["history_1"].map(markov_1_best_target_by_history_1)
    unseen_mask = pred_target.isna()
    pred_target = pred_target.fillna(markov_1_fallback_target)

    pred_encoded = pred_target.map(target_to_id)
    pred_encoded = pred_encoded.fillna(markov_1_fallback_class_id).astype(int)

    return pred_encoded, unseen_mask


y_val_pred_markov_1, val_unseen_mask_markov_1 = predict_markov_1(val_df)
y_test_pred_markov_1, test_unseen_mask_markov_1 = predict_markov_1(test_df)

display(markov_1_transition_df.head(20))
display(
    pd.DataFrame(
        {
            "split": ["validation", "test"],
            "rows": [len(val_df), len(test_df)],
            "unseen_history_1_rows": [
                int(val_unseen_mask_markov_1.sum()),
                int(test_unseen_mask_markov_1.sum()),
            ],
            "unseen_history_1_share": [
                float(val_unseen_mask_markov_1.mean()),
                float(test_unseen_mask_markov_1.mean()),
            ],
        }
    )
)

,history_1,target,count,probability,target_encoded
0,corn,soybeans,3536274,0.452430,7
1,corn,corn,3058927,0.391359,0
2,corn,forage_hay,383048,0.049007,3
3,corn,wheat,316706,0.040519,8
4,corn,fallow,138734,0.017750,2
5,corn,sorghum,114633,0.014666,6
6,corn,cotton,111504,0.014266,1
7,corn,other_cereals,97052,0.012417,5
8,corn,legumes,59294,0.007586,4
9,cotton,cotton,777655,0.626683,1


,split,rows,unseen_history_1_rows,unseen_history_1_share
0,validation,5776530,0,0.0
1,test,5776127,0,0.0


### 8.4.5 Сводка метрик baseline
Считаем accuracy, macro F1 и weighted F1 для всех трех baseline на validation и test.

In [142]:
# 8.4.5 Calculate baseline metrics summary table
baseline_rows = []
baseline_reports = {}
baseline_confusion_matrices = {}

evaluation_tasks = [
    ("most_frequent", "validation", y_val, y_val_pred_most_frequent),
    ("most_frequent", "test", y_test, y_test_pred_most_frequent),
    ("last_crop", "validation", y_val, y_val_pred_last_crop),
    ("last_crop", "test", y_test, y_test_pred_last_crop),
    ("markov_1", "validation", y_val, y_val_pred_markov_1),
    ("markov_1", "test", y_test, y_test_pred_markov_1),
]

for model_name, split_name, y_true_split, y_pred_split in evaluation_tasks:
    metrics_row, report_text, cm_df = evaluate_baseline(
        y_true=y_true_split,
        y_pred=y_pred_split,
        model_name=model_name,
        split_name=split_name,
        id_to_target=id_to_target,
    )
    baseline_rows.append(metrics_row)
    baseline_reports[(model_name, split_name)] = report_text
    baseline_confusion_matrices[(model_name, split_name)] = cm_df

baseline_results_df = (
    pd.DataFrame(baseline_rows)
    .sort_values(["split", "macro_f1", "accuracy"], ascending=[True, False, False])
    .reset_index(drop=True)
)

print("Baseline metrics summary:")
display(baseline_results_df)

Baseline metrics summary:


,model,split,accuracy,macro_f1,weighted_f1
0,markov_1,test,0.474650,0.368343,0.457863
1,last_crop,test,0.360437,0.345564,0.361570
2,most_frequent,test,0.305216,0.051965,0.142745
3,markov_1,validation,0.475235,0.368585,0.458388
4,last_crop,validation,0.359499,0.345118,0.360629
5,most_frequent,validation,0.304936,0.051929,0.142514


### 8.4.6 Детальные отчеты baseline
Печатаем classification report и confusion matrix по каждому baseline на validation и test, затем формируем краткий вывод по test.

In [ ]:
# 8.4.6 Detailed baseline reports, confusion matrices, and final comparison
for model_name, split_name in [
    ("most_frequent", "validation"),
    ("most_frequent", "test"),
    ("last_crop", "validation"),
    ("last_crop", "test"),
    ("markov_1", "validation"),
    ("markov_1", "test"),
]:
    print(f"\n=== {model_name} | {split_name} ===")
    print("Classification report:")
    print(baseline_reports[(model_name, split_name)])
    print("Confusion matrix:")
    display(baseline_confusion_matrices[(model_name, split_name)])

test_scores = baseline_results_df[baseline_results_df["split"] == "test"].copy()
best_test_row = test_scores.sort_values(
    ["macro_f1", "accuracy", "weighted_f1"], ascending=False
).iloc[0]

print(
    "\nConclusion:"
    f" stronger baseline on test by macro_f1 is '{best_test_row['model']}' "
    f"(macro_f1={best_test_row['macro_f1']:.6f}, "
    f"accuracy={best_test_row['accuracy']:.6f}, "
    f"weighted_f1={best_test_row['weighted_f1']:.6f})."
)


=== most_frequent | validation ===
Classification report:
               precision    recall  f1-score   support

         corn       0.30      1.00      0.47   1761473
       cotton       0.00      0.00      0.00    278886
       fallow       0.00      0.00      0.00    271856
   forage_hay       0.00      0.00      0.00    559815
      legumes       0.00      0.00      0.00     86986
other_cereals       0.00      0.00      0.00    150459
      sorghum       0.00      0.00      0.00    174736
     soybeans       0.00      0.00      0.00   1642055
        wheat       0.00      0.00      0.00    850264

     accuracy                           0.30   5776530
    macro avg       0.03      0.11      0.05   5776530
 weighted avg       0.09      0.30      0.14   5776530

Confusion matrix:


,pred_corn,pred_cotton,pred_fallow,pred_forage_hay,pred_legumes,pred_other_cereals,pred_sorghum,pred_soybeans,pred_wheat
true_corn,1761473,0,0,0,0,0,0,0,0
true_cotton,278886,0,0,0,0,0,0,0,0
true_fallow,271856,0,0,0,0,0,0,0,0
true_forage_hay,559815,0,0,0,0,0,0,0,0
true_legumes,86986,0,0,0,0,0,0,0,0
true_other_cereals,150459,0,0,0,0,0,0,0,0
true_sorghum,174736,0,0,0,0,0,0,0,0
true_soybeans,1642055,0,0,0,0,0,0,0,0
true_wheat,850264,0,0,0,0,0,0,0,0



=== most_frequent | test ===
Classification report:
               precision    recall  f1-score   support

         corn       0.31      1.00      0.47   1762966
       cotton       0.00      0.00      0.00    279891
       fallow       0.00      0.00      0.00    269493
   forage_hay       0.00      0.00      0.00    560068
      legumes       0.00      0.00      0.00     87287
other_cereals       0.00      0.00      0.00    151499
      sorghum       0.00      0.00      0.00    174030
     soybeans       0.00      0.00      0.00   1642434
        wheat       0.00      0.00      0.00    848459

     accuracy                           0.31   5776127
    macro avg       0.03      0.11      0.05   5776127
 weighted avg       0.09      0.31      0.14   5776127

Confusion matrix:


,pred_corn,pred_cotton,pred_fallow,pred_forage_hay,pred_legumes,pred_other_cereals,pred_sorghum,pred_soybeans,pred_wheat
true_corn,1762966,0,0,0,0,0,0,0,0
true_cotton,279891,0,0,0,0,0,0,0,0
true_fallow,269493,0,0,0,0,0,0,0,0
true_forage_hay,560068,0,0,0,0,0,0,0,0
true_legumes,87287,0,0,0,0,0,0,0,0
true_other_cereals,151499,0,0,0,0,0,0,0,0
true_sorghum,174030,0,0,0,0,0,0,0,0
true_soybeans,1642434,0,0,0,0,0,0,0,0
true_wheat,848459,0,0,0,0,0,0,0,0



=== last_crop | validation ===
Classification report:
               precision    recall  f1-score   support

         corn       0.29      0.32      0.30   1761473
       cotton       0.63      0.57      0.60    278886
       fallow       0.11      0.14      0.12    271856
   forage_hay       0.82      0.83      0.83    559815
      legumes       0.03      0.03      0.03     86986
other_cereals       0.29      0.26      0.28    150459
      sorghum       0.28      0.26      0.27    174736
     soybeans       0.29      0.27      0.28   1642055
        wheat       0.41      0.38      0.40    850264

     accuracy                           0.36   5776530
    macro avg       0.35      0.34      0.35   5776530
 weighted avg       0.36      0.36      0.36   5776530

Confusion matrix:


,pred_corn,pred_cotton,pred_fallow,pred_forage_hay,pred_legumes,pred_other_cereals,pred_sorghum,pred_soybeans,pred_wheat
true_corn,556126,30474,45583,55256,13465,19497,21460,907055,112557
true_cotton,60449,159721,4083,1396,111,1070,17429,13928,20699
true_fallow,65682,5330,37382,3198,1223,15167,38604,21629,83641
true_forage_hay,35745,1154,6447,466048,1679,15399,1737,12845,18761
true_legumes,26305,214,1625,1854,2515,5295,275,2516,46387
true_other_cereals,36229,1565,9084,10311,4300,39350,4738,10913,33969
true_sorghum,24725,18561,9561,2215,177,4017,46184,10364,58932
true_soybeans,997773,15259,58636,12312,3823,7681,14968,446169,85434
true_wheat,117550,21628,156186,15644,46884,28029,19733,121448,323162



=== last_crop | test ===
Classification report:
               precision    recall  f1-score   support

         corn       0.29      0.32      0.30   1762966
       cotton       0.63      0.57      0.60    279891
       fallow       0.11      0.14      0.12    269493
   forage_hay       0.82      0.83      0.83    560068
      legumes       0.03      0.03      0.03     87287
other_cereals       0.30      0.27      0.28    151499
      sorghum       0.28      0.26      0.27    174030
     soybeans       0.29      0.27      0.28   1642434
        wheat       0.41      0.38      0.40    848459

     accuracy                           0.36   5776127
    macro avg       0.35      0.34      0.35   5776127
 weighted avg       0.36      0.36      0.36   5776127

Confusion matrix:


,pred_corn,pred_cotton,pred_fallow,pred_forage_hay,pred_legumes,pred_other_cereals,pred_sorghum,pred_soybeans,pred_wheat
true_corn,560320,30514,45706,55338,13178,19651,21315,904622,112322
true_cotton,61235,159714,4020,1460,134,1171,17251,14237,20669
true_fallow,65189,5494,36626,3152,1197,14876,37926,21419,83614
true_forage_hay,35720,1231,6531,466047,1653,15073,1730,13068,19015
true_legumes,26099,217,1607,1770,2440,5351,308,2563,46932
true_other_cereals,36581,1668,8911,10252,4305,40263,4872,10857,33790
true_sorghum,24848,18440,9415,2208,183,4030,46102,10404,58400
true_soybeans,995771,15410,58609,12532,3832,7641,15016,448306,85317
true_wheat,117538,21555,155345,15806,47355,27835,19541,121374,322110



Conclusion: stronger baseline on test by macro_f1 is 'last_crop' (macro_f1=0.345564, accuracy=0.360437, weighted_f1=0.361570).


## Этап 9. CatBoost baseline
После эвристик обучаем CatBoost и анализируем качество на validation и test в отдельных компактных шагах.

In [ ]:
# 9.1 Train CatBoost baseline model
import importlib
from pathlib import Path

catboost_mod = importlib.import_module("catboost")
CatBoostClassifier = catboost_mod.CatBoostClassifier

cat_feature_indices = [feature_columns.index(col) for col in categorical_features]

baseline_train_dir = Path("artifacts") / "catboost" / "baseline" / "catboost_info"
baseline_train_dir.mkdir(parents=True, exist_ok=True)

cat_model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1:average=Macro",
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    random_seed=RANDOM_STATE,
    early_stopping_rounds=50,
    task_type="GPU",
    devices="0",
    train_dir=baseline_train_dir.as_posix(),
    verbose=100,
)

cat_model.fit(
    X_train,
    y_train,
    cat_features=cat_feature_indices,
    eval_set=(X_val, y_val),
    use_best_model=True,
)

print("Training completed.")
print("Best iteration:", cat_model.get_best_iteration())
print("Best validation score:", cat_model.get_best_score())
print("CatBoost train_dir:", baseline_train_dir.resolve())

0:	learn: 0.3739087	test: 0.3742636	best: 0.3742636 (0)	total: 8.27s	remaining: 2h 17m 38s
100:	learn: 0.5221245	test: 0.5225365	best: 0.5225365 (100)	total: 49.7s	remaining: 7m 22s
200:	learn: 0.5398353	test: 0.5401158	best: 0.5401402 (199)	total: 1m 30s	remaining: 6m
300:	learn: 0.5468730	test: 0.5470200	best: 0.5470200 (300)	total: 2m 12s	remaining: 5m 7s
400:	learn: 0.5512842	test: 0.5514179	best: 0.5514179 (400)	total: 2m 53s	remaining: 4m 18s
500:	learn: 0.5564240	test: 0.5566074	best: 0.5566074 (500)	total: 3m 33s	remaining: 3m 32s
600:	learn: 0.5597789	test: 0.5599470	best: 0.5599470 (600)	total: 4m 13s	remaining: 2m 48s
700:	learn: 0.5624531	test: 0.5626008	best: 0.5626008 (700)	total: 4m 55s	remaining: 2m 5s
800:	learn: 0.5651259	test: 0.5651568	best: 0.5651568 (800)	total: 5m 37s	remaining: 1m 23s
900:	learn: 0.5674736	test: 0.5672773	best: 0.5672773 (900)	total: 6m 19s	remaining: 41.7s
999:	learn: 0.5691669	test: 0.5689703	best: 0.5689703 (999)	total: 7m	remaining: 0us
best

### 9.1.1 Сохранение артефактов
Сохраняем модель и метаданные сразу после обучения, чтобы не потерять baseline при перезапуске ядра.

In [ ]:
# Save trained CatBoost model and metadata (run after training)
from pathlib import Path
import json

if "cat_model" not in globals():
    raise ValueError("cat_model was not found. Run the training cell first.")

baseline_artifacts_dir = Path("artifacts") / "catboost" / "baseline"
baseline_artifacts_dir.mkdir(parents=True, exist_ok=True)

model_path = baseline_artifacts_dir / "model.cbm"
meta_path = baseline_artifacts_dir / "meta.json"

# Save model
cat_model.save_model(model_path.as_posix())
print("Model saved to:", model_path.resolve())

# Save preprocessing/meta artifacts for reproducible inference
model_meta = {
    "feature_columns": feature_columns,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "target_to_id": target_to_id,
    "id_to_target": {str(k): v for k, v in id_to_target.items()},
}
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(model_meta, f, ensure_ascii=False, indent=2)
print("Metadata saved to:", meta_path.resolve())

Model saved to: C:\Users\Dmitry\code-projects\diploma-crop-rotation\artifacts\catboost_baseline.cbm
Metadata saved to: C:\Users\Dmitry\code-projects\diploma-crop-rotation\artifacts\catboost_baseline_meta.json


In [135]:
# 9.2 Scalar metrics on validation and test
import importlib
import numpy as np

metrics_mod = importlib.import_module("sklearn.metrics")
accuracy_score = metrics_mod.accuracy_score
f1_score = metrics_mod.f1_score
classification_report = metrics_mod.classification_report
confusion_matrix = metrics_mod.confusion_matrix

def flatten_catboost_preds(pred):
    return np.asarray(pred).reshape(-1).astype(int)

val_pred = flatten_catboost_preds(cat_model.predict(X_val))
test_pred = flatten_catboost_preds(cat_model.predict(X_test))

val_acc = accuracy_score(y_val, val_pred)
test_acc = accuracy_score(y_test, test_pred)

val_f1_macro = f1_score(y_val, val_pred, average="macro")
test_f1_macro = f1_score(y_test, test_pred, average="macro")

val_f1_weighted = f1_score(y_val, val_pred, average="weighted")
test_f1_weighted = f1_score(y_test, test_pred, average="weighted")

print("Validation metrics:")
print("accuracy:", f"{float(val_acc):.6f}")
print("macro F1:", f"{float(val_f1_macro):.6f}")
print("weighted F1:", f"{float(val_f1_weighted):.6f}")

print("\nTest metrics:")
print("accuracy:", f"{float(test_acc):.6f}")
print("macro F1:", f"{float(test_f1_macro):.6f}")
print("weighted F1:", f"{float(test_f1_weighted):.6f}")

Validation metrics:
accuracy: 0.699374
macro F1: 0.568970
weighted F1: 0.690008

Test metrics:
accuracy: 0.699023
macro F1: 0.568700
weighted F1: 0.689667


In [136]:
# 9.3 Classification reports and confusion matrix
labels_sorted = sorted(id_to_target.keys())
target_names = [id_to_target[i] for i in labels_sorted]

print("Classification report (validation):")
print(
    classification_report(
        y_val,
        val_pred,
        labels=labels_sorted,
        target_names=target_names,
        zero_division=0,
    )
)

print("Classification report (test):")
print(
    classification_report(
        y_test,
        test_pred,
        labels=labels_sorted,
        target_names=target_names,
        zero_division=0,
    )
)

cm_test = confusion_matrix(y_test, test_pred, labels=labels_sorted)
cm_test_df = pd.DataFrame(
    cm_test,
    index=[f"true_{id_to_target[i]}" for i in labels_sorted],
    columns=[f"pred_{id_to_target[i]}" for i in labels_sorted],
)

print("Confusion matrix (test):")
display(cm_test_df)

print("Target classes in model order:")
display(pd.DataFrame({"target": list(target_to_id.keys()), "target_id": list(target_to_id.values())}))

Classification report (validation):
               precision    recall  f1-score   support

         corn       0.69      0.72      0.71   1761473
       cotton       0.72      0.77      0.75    278886
       fallow       0.59      0.41      0.48    271856
   forage_hay       0.81      0.85      0.83    559815
      legumes       0.51      0.14      0.22     86986
other_cereals       0.53      0.24      0.33    150459
      sorghum       0.50      0.31      0.38    174736
     soybeans       0.72      0.76      0.74   1642055
        wheat       0.66      0.73      0.69    850264

     accuracy                           0.70   5776530
    macro avg       0.64      0.55      0.57   5776530
 weighted avg       0.69      0.70      0.69   5776530

Classification report (test):
               precision    recall  f1-score   support

         corn       0.69      0.72      0.71   1762966
       cotton       0.72      0.77      0.75    279891
       fallow       0.59      0.41      0.48    26

,pred_corn,pred_cotton,pred_fallow,pred_forage_hay,pred_legumes,pred_other_cereals,pred_sorghum,pred_soybeans,pred_wheat
true_corn,1266907,26275,18084,58603,2478,5825,12452,317801,54541
true_cotton,16478,214947,2268,1567,23,372,6892,18202,19142
true_fallow,34420,5289,109781,4167,922,4257,11748,45381,53528
true_forage_hay,43206,1747,2210,476321,444,5736,642,14023,15739
true_legumes,12377,292,1920,2435,11815,2008,176,16091,40173
true_other_cereals,29578,2405,9550,12508,559,37102,2935,11159,45703
true_sorghum,30125,13929,17983,2341,49,2243,53063,9580,44717
true_soybeans,306646,13902,4814,12787,3191,944,3173,1249492,47485
true_wheat,89510,17932,20892,17683,4054,10952,16622,52599,618215


Target classes in model order:


,target,target_id
0,corn,0
1,cotton,1
2,fallow,2
3,forage_hay,3
4,legumes,4
5,other_cereals,5
6,sorghum,6
7,soybeans,7
8,wheat,8


In [137]:
# 9.4 Top-k prediction examples for interpretation
test_pred_decoded = pd.Series(test_pred, index=y_test.index).map(id_to_target)
y_test_decoded = y_test.map(id_to_target)

test_proba = cat_model.predict_proba(X_test)
model_classes = np.asarray(cat_model.classes_).astype(int)

top_k = 3
n_examples = min(10, len(X_test))
top_k_positions = np.argsort(test_proba, axis=1)[:, -top_k:][:, ::-1]

examples = []
for row_pos in range(n_examples):
    row = {
        "row_index": X_test.index[row_pos],
        "true_class": y_test_decoded.iloc[row_pos],
        "pred_class": test_pred_decoded.iloc[row_pos],
    }
    for rank in range(top_k):
        class_pos = top_k_positions[row_pos, rank]
        class_id = int(model_classes[class_pos])
        row[f"top_{rank + 1}_class"] = id_to_target.get(class_id, str(class_id))
        row[f"top_{rank + 1}_proba"] = float(test_proba[row_pos, class_pos])
    examples.append(row)

prediction_examples = pd.DataFrame(examples)
print("Prediction examples with top-k probabilities:")
display(prediction_examples)

Prediction examples with top-k probabilities:


,row_index,true_class,pred_class,top_1_class,top_1_proba,top_2_class,top_2_proba,top_3_class,top_3_proba
0,6,sorghum,forage_hay,forage_hay,0.363198,other_cereals,0.237893,wheat,0.118387
1,8,fallow,forage_hay,forage_hay,0.343382,other_cereals,0.246198,wheat,0.126503
2,11,sorghum,forage_hay,forage_hay,0.340912,other_cereals,0.246477,wheat,0.126584
3,13,wheat,wheat,wheat,0.288960,forage_hay,0.231117,sorghum,0.160771
4,16,sorghum,sorghum,sorghum,0.324745,fallow,0.268763,wheat,0.199667
5,28,sorghum,sorghum,sorghum,0.323262,fallow,0.264587,wheat,0.204683
6,40,forage_hay,forage_hay,forage_hay,0.844142,wheat,0.041776,corn,0.029447
7,43,sorghum,wheat,wheat,0.298559,fallow,0.281713,sorghum,0.189582
8,46,wheat,sorghum,sorghum,0.359379,fallow,0.303201,wheat,0.214025
9,66,wheat,wheat,wheat,0.535111,sorghum,0.229770,fallow,0.116625


### 9.5 Итоговое сравнение моделей
Объединяем baseline-результаты и CatBoost в единую таблицу для финального сравнения на validation и test.

In [143]:
# 9.5 Final comparison table: heuristics + CatBoost
if "baseline_results_df" not in globals():
    raise ValueError("baseline_results_df was not found. Run baseline metrics cells first.")

required_catboost_metrics = [
    "val_acc",
    "val_f1_macro",
    "val_f1_weighted",
    "test_acc",
    "test_f1_macro",
    "test_f1_weighted",
]
missing_catboost_metrics = [name for name in required_catboost_metrics if name not in globals()]
if missing_catboost_metrics:
    raise ValueError(
        "Missing CatBoost metrics variables: "
        + ", ".join(missing_catboost_metrics)
        + ". Run CatBoost metrics cell first."
    )

catboost_results_df = pd.DataFrame(
    [
        {
            "model": "catboost",
            "split": "validation",
            "accuracy": float(val_acc),
            "macro_f1": float(val_f1_macro),
            "weighted_f1": float(val_f1_weighted),
        },
        {
            "model": "catboost",
            "split": "test",
            "accuracy": float(test_acc),
            "macro_f1": float(test_f1_macro),
            "weighted_f1": float(test_f1_weighted),
        },
    ]
)

base_models = ["most_frequent", "last_crop", "markov_1"]
base_results_df = baseline_results_df[baseline_results_df["model"].isin(base_models)].copy()

results_df = pd.concat(
    [
        base_results_df[["model", "split", "accuracy", "macro_f1", "weighted_f1"]],
        catboost_results_df,
    ],
    ignore_index=True,
)

results_df["split"] = pd.Categorical(results_df["split"], categories=["validation", "test"], ordered=True)
results_df = (
    results_df
    .sort_values(["split", "accuracy"], ascending=[True, False])
    .reset_index(drop=True)
)

display(results_df)

,model,split,accuracy,macro_f1,weighted_f1
0,catboost,validation,0.699374,0.568970,0.690008
1,markov_1,validation,0.475235,0.368585,0.458388
2,last_crop,validation,0.359499,0.345118,0.360629
3,most_frequent,validation,0.304936,0.051929,0.142514
4,catboost,test,0.699023,0.568700,0.689667
5,markov_1,test,0.474650,0.368343,0.457863
6,last_crop,test,0.360437,0.345564,0.361570
7,most_frequent,test,0.305216,0.051965,0.142745


## Этап 10. Загрузка и повторное использование модели
При необходимости загружаем сохраненные артефакты и выполняем инференс без повторного обучения CatBoost.

### 10.1 Загрузка и инференс
Используем сохраненные артефакты, чтобы быстро получить предсказания без повторного обучения CatBoost.

In [ ]:
# Load saved CatBoost model and run inference
from pathlib import Path
import json
import importlib
import numpy as np
import pandas as pd

baseline_artifacts_dir = Path("artifacts") / "catboost" / "baseline"
model_path = baseline_artifacts_dir / "model.cbm"
meta_path = baseline_artifacts_dir / "meta.json"

if not model_path.exists() or not meta_path.exists():
    raise FileNotFoundError(
        f"Saved artifacts were not found. Expected files: {model_path} and {meta_path}"
    )

catboost_mod = importlib.import_module("catboost")
CatBoostClassifier = catboost_mod.CatBoostClassifier

loaded_cat_model = CatBoostClassifier()
loaded_cat_model.load_model(model_path.as_posix())
print("Loaded model from:", model_path.resolve())

with open(meta_path, "r", encoding="utf-8") as f:
    loaded_meta = json.load(f)
print("Loaded metadata keys:", list(loaded_meta.keys()))

loaded_feature_columns = loaded_meta["feature_columns"]
id_to_target_loaded = {int(k): v for k, v in loaded_meta["id_to_target"].items()}

if "X_test" in globals():
    X_infer = X_test[loaded_feature_columns].copy()
    infer_source = "X_test"
else:
    if "df_model" not in globals():
        raise ValueError("X_test and df_model are not available for inference.")
    X_infer = df_model[loaded_feature_columns].head(1000).copy()
    infer_source = "df_model.head(1000)"

pred_ids = np.asarray(loaded_cat_model.predict(X_infer)).reshape(-1).astype(int)
pred_labels = pd.Series(pred_ids).map(id_to_target_loaded)

print("Inference source:", infer_source)
print("Rows predicted:", len(X_infer))

prediction_preview = pd.DataFrame({
    "pred_id": pred_ids,
    "pred_label": pred_labels,
})
display(prediction_preview.head(20))

# Optional quality check when test labels exist
if "y_test" in globals() and infer_source == "X_test":
    metrics_mod = importlib.import_module("sklearn.metrics")
    accuracy_score = metrics_mod.accuracy_score
    test_acc_loaded = accuracy_score(y_test, pred_ids)
    print("Loaded model test accuracy:", f"{float(test_acc_loaded):.6f}")

## Этап 11. Improved CatBoost с derived features
Добавляем derived features поверх существующего dataframe модели, переиспользуем текущий split по CSBID и готовим improved CatBoost

### 11.1 Feature engineering на копии данных
Создаем df_features как копию существующего dataframe, добавляем derived features первого уровня и делаем sanity-check.

In [32]:
# 11.1 Feature engineering: copy existing modeling dataframe and add derived features
import numpy as np
import pandas as pd

if "df_model" in globals():
    df_features = df_model.copy()
    source_df_name = "df_model"
elif "window_df_filtered" in globals():
    df_features = window_df_filtered.copy()
    source_df_name = "window_df_filtered"
else:
    raise ValueError("Neither df_model nor window_df_filtered is available in globals().")

required_columns = [
    "CSBID",
    "target_year",
    "history_1",
    "history_2",
    "history_3",
    "target",
    "CSBACRES",
    "STATEFIPS",
    "CNTYFIPS",
    "INSIDE_X",
    "INSIDE_Y",
]
missing_required = [col for col in required_columns if col not in df_features.columns]
if missing_required:
    raise ValueError(f"Missing required columns in {source_df_name}: {missing_required}")

history_cols = ["history_1", "history_2", "history_3"]
df_features[history_cols] = df_features[history_cols].astype("string")
df_features["target"] = df_features["target"].astype("string")

h1 = df_features["history_1"]
h2 = df_features["history_2"]
h3 = df_features["history_3"]

# Fast path for the common case without missing history values
if not df_features[history_cols].isna().any(axis=None):
    df_features["n_unique_history"] = np.select(
        [
            (h1 == h2) & (h2 == h3),                 # AAA
            (h1 == h2) | (h1 == h3) | (h2 == h3),   # AAB / ABA / ABB
        ],
        [1, 2],
        default=3,
    ).astype("int8")
else:
    # Fallback keeps exact dropna=True behavior when missing values are present
    df_features["n_unique_history"] = (
        df_features[history_cols].nunique(axis=1, dropna=True).astype("int8")
    )

df_features["has_fallow_in_history"] = (df_features[history_cols] == "fallow").any(axis=1).astype("int8")
df_features["has_legume_in_history"] = (df_features[history_cols] == "legumes").any(axis=1).astype("int8")
df_features["same_h1_h2"] = (h1 == h2).astype("int8")
df_features["same_h2_h3"] = (h2 == h3).astype("int8")
df_features["same_all_history"] = (
    (h1 == h2)
    & (h2 == h3)
).astype("int8")

derived_feature_cols = [
    "n_unique_history",
    "has_fallow_in_history",
    "has_legume_in_history",
    "same_h1_h2",
    "same_h2_h3",
    "same_all_history",
]

print("Source dataframe:", source_df_name)
print("df_features shape:", df_features.shape)
print("New derived columns:", derived_feature_cols)

preview_cols = history_cols + derived_feature_cols
display(df_features[preview_cols].head(10))

for col in ["has_fallow_in_history", "has_legume_in_history", "same_h1_h2", "same_h2_h3", "same_all_history"]:
    print(f"\nvalue_counts for {col}:")
    display(df_features[col].value_counts(dropna=False).sort_index())

print("\nn_unique_history describe:")
display(df_features["n_unique_history"].describe())

Source dataframe: df_model
df_features shape: (38511210, 18)
New derived columns: ['n_unique_history', 'has_fallow_in_history', 'has_legume_in_history', 'same_h1_h2', 'same_h2_h3', 'same_all_history']


,history_1,history_2,history_3,n_unique_history,has_fallow_in_history,has_legume_in_history,same_h1_h2,same_h2_h3,same_all_history
0,wheat,wheat,sorghum,2,0,0,1,0,0
1,wheat,wheat,sorghum,2,0,0,1,0,0
2,forage_hay,forage_hay,other_cereals,2,0,0,1,0,0
3,forage_hay,forage_hay,other_cereals,2,0,0,1,0,0
4,forage_hay,forage_hay,other_cereals,2,0,0,1,0,0
5,forage_hay,forage_hay,other_cereals,2,0,0,1,0,0
6,forage_hay,forage_hay,other_cereals,2,0,0,1,0,0
7,forage_hay,forage_hay,wheat,2,0,0,1,0,0
8,forage_hay,forage_hay,other_cereals,2,0,0,1,0,0
9,forage_hay,forage_hay,wheat,2,0,0,1,0,0



value_counts for has_fallow_in_history:


has_fallow_in_history
0    32905153
1     5606057
Name: count, dtype: int64


value_counts for has_legume_in_history:


...

same_h2_h3
0    25021183
1    13490027
Name: count, dtype: int64


value_counts for same_all_history:


same_all_history
0    29746077
1     8765133
Name: count, dtype: int64


n_unique_history describe:


count    3.851121e+07
mean     1.940935e+00
std      6.266142e-01
min      1.000000e+00
25%      2.000000e+00
50%      2.000000e+00
75%      2.000000e+00
max      3.000000e+00
Name: n_unique_history, dtype: float64

In [33]:
# 11.2 Feature lists for improved CatBoost
improved_cat_features = [
    "history_1",
    "history_2",
    "history_3",
    "STATEFIPS",
    "CNTYFIPS",
]

improved_num_features = [
    "CSBACRES",
    "INSIDE_X",
    "INSIDE_Y",
    "n_unique_history",
    "has_fallow_in_history",
    "has_legume_in_history",
    "same_h1_h2",
    "same_h2_h3",
    "same_all_history",
]

missing_improved_features = [
    col for col in (improved_cat_features + improved_num_features)
    if col not in df_features.columns
]
if missing_improved_features:
    raise ValueError(f"Missing improved feature columns: {missing_improved_features}")

improved_feature_cols = improved_cat_features + improved_num_features

print("improved_cat_features:", improved_cat_features)
print("improved_num_features:", improved_num_features)
print("improved_feature_cols:", improved_feature_cols)

display(
    pd.DataFrame(
        {
            "feature": improved_feature_cols,
            "feature_type": ["categorical"] * len(improved_cat_features)
            + ["numeric"] * len(improved_num_features),
        }
    )
)

improved_cat_features: ['history_1', 'history_2', 'history_3', 'STATEFIPS', 'CNTYFIPS']
improved_num_features: ['CSBACRES', 'INSIDE_X', 'INSIDE_Y', 'n_unique_history', 'has_fallow_in_history', 'has_legume_in_history', 'same_h1_h2', 'same_h2_h3', 'same_all_history']
improved_feature_cols: ['history_1', 'history_2', 'history_3', 'STATEFIPS', 'CNTYFIPS', 'CSBACRES', 'INSIDE_X', 'INSIDE_Y', 'n_unique_history', 'has_fallow_in_history', 'has_legume_in_history', 'same_h1_h2', 'same_h2_h3', 'same_all_history']


,feature,feature_type
0,history_1,categorical
1,history_2,categorical
2,history_3,categorical
3,STATEFIPS,categorical
4,CNTYFIPS,categorical
5,CSBACRES,numeric
6,INSIDE_X,numeric
7,INSIDE_Y,numeric
8,n_unique_history,numeric
9,has_fallow_in_history,numeric


### 11.3 Переиспользование текущего split и подготовка матриц
Используем существующий split по CSBID (train_ids/val_ids/test_ids или train_df/val_df/test_df), не делаем новый случайный split по строкам.

In [34]:
# 11.3 Reuse existing CSBID split and prepare X/y for improved CatBoost
# Keep target as string and create/use encoded target separately
if "target_encoded" in df_features.columns:
    improved_target_col = "target_encoded"
    df_features[improved_target_col] = pd.to_numeric(
        df_features[improved_target_col], errors="raise"
    ).astype(int)

    if "target_to_id" in globals() and "id_to_target" in globals():
        target_to_id_improved = {str(k): int(v) for k, v in target_to_id.items()}
        id_to_target_improved = {int(k): str(v) for k, v in id_to_target.items()}
    else:
        mapping_df_local = (
            df_features[["target", improved_target_col]]
            .dropna()
            .drop_duplicates()
            .sort_values(improved_target_col)
        )
        target_to_id_improved = dict(
            zip(
                mapping_df_local["target"].astype(str),
                mapping_df_local[improved_target_col].astype(int),
            )
        )
        id_to_target_improved = dict(
            zip(
                mapping_df_local[improved_target_col].astype(int),
                mapping_df_local["target"].astype(str),
            )
        )
else:
    improved_target_col = "target_encoded_improved"
    if "target_to_id" in globals() and "id_to_target" in globals():
        target_to_id_improved = {str(k): int(v) for k, v in target_to_id.items()}
        id_to_target_improved = {int(k): str(v) for k, v in id_to_target.items()}
    else:
        class_labels = sorted(df_features["target"].dropna().astype(str).unique().tolist())
        target_to_id_improved = {label: idx for idx, label in enumerate(class_labels)}
        id_to_target_improved = {idx: label for label, idx in target_to_id_improved.items()}

    df_features[improved_target_col] = df_features["target"].map(target_to_id_improved)

    unmapped_target_mask = df_features[improved_target_col].isna()
    if unmapped_target_mask.any():
        unmapped_targets = (
            df_features.loc[unmapped_target_mask, "target"]
            .astype(str)
            .value_counts()
            .head(20)
        )
        raise ValueError(
            "Some target values were not mapped to encoded ids. "
            f"Top unmapped values: {unmapped_targets.to_dict()}"
        )

    df_features[improved_target_col] = df_features[improved_target_col].astype(int)

print("Encoded target column:", improved_target_col)
print("Unique target classes:", len(target_to_id_improved))

# Reuse existing split ids by CSBID
if all(name in globals() for name in ["train_ids", "val_ids", "test_ids"]):
    split_source = "train_ids/val_ids/test_ids"
    train_ids_improved = pd.Series(train_ids).astype("string")
    val_ids_improved = pd.Series(val_ids).astype("string")
    test_ids_improved = pd.Series(test_ids).astype("string")
elif all(name in globals() for name in ["train_df", "val_df", "test_df"]):
    split_source = "train_df/val_df/test_df"
    train_ids_improved = pd.Series(train_df["CSBID"].dropna().unique()).astype("string")
    val_ids_improved = pd.Series(val_df["CSBID"].dropna().unique()).astype("string")
    test_ids_improved = pd.Series(test_df["CSBID"].dropna().unique()).astype("string")
else:
    raise ValueError(
        "Split ids were not found. Expected train_ids/val_ids/test_ids or train_df/val_df/test_df in globals()."
    )

csbid_key = df_features["CSBID"].astype("string")
train_id_set_improved = set(train_ids_improved.dropna().tolist())
val_id_set_improved = set(val_ids_improved.dropna().tolist())
test_id_set_improved = set(test_ids_improved.dropna().tolist())

train_df_improved = df_features.loc[csbid_key.isin(train_id_set_improved)].copy()
val_df_improved = df_features.loc[csbid_key.isin(val_id_set_improved)].copy()
test_df_improved = df_features.loc[csbid_key.isin(test_id_set_improved)].copy()

print("Split source:", split_source)
print("Rows by split:")
print("train_df_improved:", train_df_improved.shape)
print("val_df_improved:", val_df_improved.shape)
print("test_df_improved:", test_df_improved.shape)

# Build feature matrices and targets
X_train_improved = train_df_improved[improved_feature_cols].copy()
X_val_improved = val_df_improved[improved_feature_cols].copy()
X_test_improved = test_df_improved[improved_feature_cols].copy()

for X_split in [X_train_improved, X_val_improved, X_test_improved]:
    for col in improved_cat_features:
        X_split[col] = X_split[col].astype("string")
    for col in improved_num_features:
        X_split[col] = pd.to_numeric(X_split[col], errors="coerce")

y_train_improved = train_df_improved[improved_target_col].astype(int).copy()
y_val_improved = val_df_improved[improved_target_col].astype(int).copy()
y_test_improved = test_df_improved[improved_target_col].astype(int).copy()

print("\nX/y shapes:")
print("X_train_improved:", X_train_improved.shape, "| y_train_improved:", y_train_improved.shape)
print("X_val_improved:", X_val_improved.shape, "| y_val_improved:", y_val_improved.shape)
print("X_test_improved:", X_test_improved.shape, "| y_test_improved:", y_test_improved.shape)

print("\nMissing values in improved features by split:")
display(
    pd.DataFrame(
        {
            "feature": improved_feature_cols,
            "train_missing": [int(X_train_improved[col].isna().sum()) for col in improved_feature_cols],
            "val_missing": [int(X_val_improved[col].isna().sum()) for col in improved_feature_cols],
            "test_missing": [int(X_test_improved[col].isna().sum()) for col in improved_feature_cols],
        }
    )
)

Encoded target column: target_encoded
Unique target classes: 9
Split source: train_ids/val_ids/test_ids
Rows by split:
train_df_improved: (26958553, 18)
val_df_improved: (5776530, 18)
test_df_improved: (5776127, 18)

X/y shapes:
X_train_improved: (26958553, 14) | y_train_improved: (26958553,)
X_val_improved: (5776530, 14) | y_val_improved: (5776530,)
X_test_improved: (5776127, 14) | y_test_improved: (5776127,)

Missing values in improved features by split:


,feature,train_missing,val_missing,test_missing
0,history_1,0,0,0
1,history_2,0,0,0
2,history_3,0,0,0
3,STATEFIPS,0,0,0
4,CNTYFIPS,0,0,0
5,CSBACRES,0,0,0
6,INSIDE_X,0,0,0
7,INSIDE_Y,0,0,0
8,n_unique_history,0,0,0
9,has_fallow_in_history,0,0,0


### 11.3.1 Checkpoint: стратегия сохранения
Сохраняем baseline- и improved-объекты в artifacts/research_checkpoint, чтобы после очистки RAM подгружать только нужный набор данных.

In [ ]:
# 11.3.2 Checkpoint helpers: save/load and memory cleanup
import gc
import json
from pathlib import Path
import pandas as pd

ARTIFACTS_DIR = Path("artifacts")
CHECKPOINT_DIR = ARTIFACTS_DIR / "research_checkpoint"

BASELINE_CORE_OBJECTS = [
    "window_df_filtered",  # Main table for creating new baselines
    "df_model",            # Prepared modeling table with encoded target
    "train_df",
    "val_df",
    "test_df",
    "train_ids",
    "val_ids",
    "test_ids",
]

IMPROVED_CORE_OBJECTS = [
    "X_train_improved",
    "X_val_improved",
    "X_test_improved",
    "y_train_improved",
    "y_val_improved",
    "y_test_improved",
]

def _save_pickle_objects(object_names, target_dir):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    saved = []
    skipped = []
    for name in object_names:
        if name in globals():
            file_path = target_dir / f"{name}.pkl"
            pd.to_pickle(globals()[name], file_path)
            size_mb = file_path.stat().st_size / (1024 ** 2)
            saved.append((name, size_mb))
        else:
            skipped.append(name)
    return saved, skipped

def _load_pickle_objects(target_dir):
    target_dir = Path(target_dir)
    loaded = []
    if not target_dir.exists():
        return loaded

    for file_path in sorted(target_dir.glob("*.pkl")):
        var_name = file_path.stem
        globals()[var_name] = pd.read_pickle(file_path)
        loaded.append(var_name)
    return loaded

def save_research_checkpoint(checkpoint_dir=CHECKPOINT_DIR, save_baseline_matrices=False):
    checkpoint_dir = Path(checkpoint_dir)
    baseline_dir = checkpoint_dir / "baseline"
    improved_dir = checkpoint_dir / "improved"

    baseline_objects = BASELINE_CORE_OBJECTS.copy()
    if save_baseline_matrices:
        baseline_objects += [
            "X_train",
            "X_val",
            "X_test",
            "y_train",
            "y_val",
            "y_test",
        ]

    saved_baseline, skipped_baseline = _save_pickle_objects(baseline_objects, baseline_dir)
    saved_improved, skipped_improved = _save_pickle_objects(IMPROVED_CORE_OBJECTS, improved_dir)

    baseline_meta = {
        "feature_columns": list(feature_columns) if "feature_columns" in globals() else None,
        "categorical_features": list(categorical_features) if "categorical_features" in globals() else None,
        "numeric_features": list(numeric_features) if "numeric_features" in globals() else None,
        "target_to_id": ({str(k): int(v) for k, v in target_to_id.items()} if "target_to_id" in globals() else None),
        "id_to_target": ({str(int(k)): str(v) for k, v in id_to_target.items()} if "id_to_target" in globals() else None),
        "saved_baseline_matrices": bool(save_baseline_matrices),
    }

    improved_meta = {
        "improved_feature_cols": list(improved_feature_cols) if "improved_feature_cols" in globals() else None,
        "improved_cat_features": list(improved_cat_features) if "improved_cat_features" in globals() else None,
        "improved_num_features": list(improved_num_features) if "improved_num_features" in globals() else None,
        "improved_target_col": str(improved_target_col) if "improved_target_col" in globals() else None,
        "target_to_id_improved": ({str(k): int(v) for k, v in target_to_id_improved.items()} if "target_to_id_improved" in globals() else None),
        "id_to_target_improved": ({str(int(k)): str(v) for k, v in id_to_target_improved.items()} if "id_to_target_improved" in globals() else None),
    }

    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    with open(checkpoint_dir / "baseline_meta.json", "w", encoding="utf-8") as f:
        json.dump(baseline_meta, f, ensure_ascii=False, indent=2)
    with open(checkpoint_dir / "improved_meta.json", "w", encoding="utf-8") as f:
        json.dump(improved_meta, f, ensure_ascii=False, indent=2)

    print("Checkpoint saved to:", checkpoint_dir.resolve())
    print("\nSaved baseline objects:")
    for name, size_mb in saved_baseline:
        print(f"  {name}.pkl: {size_mb:,.2f} MB")
    print("Skipped baseline objects:", skipped_baseline)

    print("\nSaved improved objects:")
    for name, size_mb in saved_improved:
        print(f"  {name}.pkl: {size_mb:,.2f} MB")
    print("Skipped improved objects:", skipped_improved)

def clear_heavy_objects(preserve_vars=None):
    if preserve_vars is None:
        preserve_vars = {"RANDOM_STATE", "ARTIFACTS_DIR", "CHECKPOINT_DIR"}

    heavy_vars = [
        "df", "df_prepared", "df_filtered", "final_df", "window_df", "window_df_filtered",
        "df_model", "df_features",
        "train_df", "val_df", "test_df",
        "train_df_improved", "val_df_improved", "test_df_improved",
        "X_train", "X_val", "X_test",
        "y_train", "y_val", "y_test",
        "X_train_improved", "X_val_improved", "X_test_improved",
        "y_train_improved", "y_val_improved", "y_test_improved",
        "summary_all_years", "summary_per_year", "target_summary", "target_summary_filtered",
        "unknown_by_year", "unknown_by_code",
        "baseline_results_df", "baseline_reports", "baseline_confusion_matrices",
        "cat_model", "improved_cat_model",
        "val_pred", "test_pred", "val_pred_improved", "test_pred_improved",
    ]

    deleted = []
    for name in heavy_vars:
        if name in globals() and name not in preserve_vars:
            del globals()[name]
            deleted.append(name)

    freed = gc.collect()
    print("Deleted variables:", len(deleted))
    print("GC collected objects:", freed)
    if deleted:
        print("Deleted list:", deleted)

def load_research_checkpoint(checkpoint_dir=CHECKPOINT_DIR, load_baseline=False, load_improved=True):
    checkpoint_dir = Path(checkpoint_dir)
    baseline_dir = checkpoint_dir / "baseline"
    improved_dir = checkpoint_dir / "improved"

    loaded_baseline = []
    loaded_improved = []

    if load_baseline:
        loaded_baseline = _load_pickle_objects(baseline_dir)
        baseline_meta_path = checkpoint_dir / "baseline_meta.json"
        if baseline_meta_path.exists():
            with open(baseline_meta_path, "r", encoding="utf-8") as f:
                meta = json.load(f)

            if meta.get("feature_columns") is not None:
                globals()["feature_columns"] = list(meta["feature_columns"])
            if meta.get("categorical_features") is not None:
                globals()["categorical_features"] = list(meta["categorical_features"])
            if meta.get("numeric_features") is not None:
                globals()["numeric_features"] = list(meta["numeric_features"])
            if meta.get("target_to_id") is not None:
                globals()["target_to_id"] = {str(k): int(v) for k, v in meta["target_to_id"].items()}
            if meta.get("id_to_target") is not None:
                globals()["id_to_target"] = {int(k): str(v) for k, v in meta["id_to_target"].items()}

    if load_improved:
        loaded_improved = _load_pickle_objects(improved_dir)
        improved_meta_path = checkpoint_dir / "improved_meta.json"
        if improved_meta_path.exists():
            with open(improved_meta_path, "r", encoding="utf-8") as f:
                meta = json.load(f)

            if meta.get("improved_feature_cols") is not None:
                globals()["improved_feature_cols"] = list(meta["improved_feature_cols"])
            if meta.get("improved_cat_features") is not None:
                globals()["improved_cat_features"] = list(meta["improved_cat_features"])
            if meta.get("improved_num_features") is not None:
                globals()["improved_num_features"] = list(meta["improved_num_features"])
            if meta.get("improved_target_col") is not None:
                globals()["improved_target_col"] = str(meta["improved_target_col"])
            if meta.get("target_to_id_improved") is not None:
                globals()["target_to_id_improved"] = {str(k): int(v) for k, v in meta["target_to_id_improved"].items()}
            if meta.get("id_to_target_improved") is not None:
                globals()["id_to_target_improved"] = {int(k): str(v) for k, v in meta["id_to_target_improved"].items()}

            if "improved_feature_cols" in globals() and "improved_cat_features" in globals():
                globals()["improved_cat_feature_indices"] = [
                    improved_feature_cols.index(col) for col in improved_cat_features
                ]

    print("Checkpoint loaded from:", checkpoint_dir.resolve())
    print("Loaded baseline objects:", loaded_baseline if load_baseline else "not requested")
    print("Loaded improved objects:", loaded_improved if load_improved else "not requested")

Checkpoint saved to: C:\Users\Dmitry\code-projects\diploma-crop-rotation\artifacts\research_checkpoint

Saved baseline objects:
  window_df_filtered.pkl: 4,263.56 MB
  df_model.pkl: 4,953.47 MB
  train_df.pkl: 3,467.61 MB
  val_df.pkl: 742.95 MB
  test_df.pkl: 742.90 MB
  train_ids.pkl: 97.17 MB
  val_ids.pkl: 20.82 MB
  test_ids.pkl: 20.82 MB
Skipped baseline objects: []

Saved improved objects:
  X_train_improved.pkl: 2,640.73 MB
  X_val_improved.pkl: 565.80 MB
  X_test_improved.pkl: 565.75 MB
  y_train_improved.pkl: 617.03 MB
  y_val_improved.pkl: 132.21 MB
  y_test_improved.pkl: 132.21 MB
Skipped improved objects: []
Deleted variables: 32
GC collected objects: 174
Deleted list: ['df', 'df_prepared', 'df_filtered', 'final_df', 'window_df', 'window_df_filtered', 'df_model', 'df_features', 'train_df', 'val_df', 'test_df', 'train_df_improved', 'val_df_improved', 'test_df_improved', 'X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test', 'X_train_improved', 'X_val_improved', 'X_test

### 11.3.3 Checkpoint: сценарий запуска
По умолчанию: сохранить baseline/improved checkpoint, очистить RAM и загрузить только improved-объекты для обучения Stage 11.

In [ ]:
# 11.3.4 Checkpoint flow: save -> clear RAM -> selective reload
# Default flow: save baseline+improved -> clear RAM -> load improved only for training.
RUN_SAVE_CHECKPOINT = True
RUN_CLEAR_RAM = True
RUN_LOAD_BASELINE_AFTER_CLEAR = False
RUN_LOAD_IMPROVED_AFTER_CLEAR = True

# Optional: also persist baseline X/y matrices (can use a lot of disk).
SAVE_BASELINE_MATRICES = False

if RUN_SAVE_CHECKPOINT:
    save_research_checkpoint(CHECKPOINT_DIR, save_baseline_matrices=SAVE_BASELINE_MATRICES)

if RUN_CLEAR_RAM:
    clear_heavy_objects()

if RUN_LOAD_BASELINE_AFTER_CLEAR or RUN_LOAD_IMPROVED_AFTER_CLEAR:
    load_research_checkpoint(
        CHECKPOINT_DIR,
        load_baseline=RUN_LOAD_BASELINE_AFTER_CLEAR,
        load_improved=RUN_LOAD_IMPROVED_AFTER_CLEAR,
    )

print("Checkpoint flow completed.")
print("For improved training now: keep RUN_LOAD_IMPROVED_AFTER_CLEAR=True and RUN_LOAD_BASELINE_AFTER_CLEAR=False.")
print("For future baselines: call load_research_checkpoint(load_baseline=True, load_improved=False).")

### 11.4 Improved CatBoost: создание модели
Инициализируем improved CatBoost с параметрами baseline-уровня

In [ ]:
# 11.4 Create improved CatBoost model (GPU-only, baseline-like)
import pandas as pd
from pathlib import Path
from catboost import CatBoostClassifier

improved_train_dir = Path("artifacts") / "catboost" / "improved" / "catboost_info"
improved_train_dir.mkdir(parents=True, exist_ok=True)

catboost_params_improved = {
    "loss_function": "MultiClass",
    "eval_metric": "TotalF1:average=Macro",
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 6,
    "l2_leaf_reg": 3.0,
    "random_seed": RANDOM_STATE if "RANDOM_STATE" in globals() else 42,
    "early_stopping_rounds": 50,
    "task_type": "GPU",
    "devices": "0",
    "train_dir": improved_train_dir.as_posix(),
    "verbose": 100,
    # Helps reduce GPU memory pressure on large data.
    "gpu_ram_part": 0.80,
    "max_ctr_complexity": 1,
}

improved_cat_feature_indices = [
    improved_feature_cols.index(col) for col in improved_cat_features
]

improved_cat_model = CatBoostClassifier(**catboost_params_improved)

print("Improved CatBoost model initialized (GPU-only).")
print("Categorical feature indices:", improved_cat_feature_indices)
print("Model params:")
display(pd.Series(catboost_params_improved, name="value"))

Improved CatBoost model initialized (GPU-only).
Categorical feature indices: [0, 1, 2, 3, 4]
Model params:


loss_function                       MultiClass
eval_metric              TotalF1:average=Macro
iterations                                1000
learning_rate                             0.05
depth                                        6
l2_leaf_reg                                3.0
random_seed                                 42
early_stopping_rounds                       50
task_type                                  GPU
devices                                      0
verbose                                    100
gpu_ram_part                               0.8
max_ctr_complexity                           1
Name: value, dtype: object

### 11.5 Обучение improved CatBoost (ручной запуск)

In [37]:
# 11.5 Train improved CatBoost model (GPU, baseline-like)
import pandas as pd
from catboost import CatBoostError

# Prepare dtypes similarly to baseline flow.
for X_split in [X_train_improved, X_val_improved]:
    for col in improved_cat_features:
        X_split[col] = X_split[col].astype("string")
    for col in improved_num_features:
        X_split[col] = pd.to_numeric(X_split[col], errors="coerce").astype("float32")

# Remove rows with missing numeric features before fit.
train_ok_mask = X_train_improved[improved_num_features].notna().all(axis=1)
val_ok_mask = X_val_improved[improved_num_features].notna().all(axis=1)

X_train_fit = X_train_improved.loc[train_ok_mask].copy()
y_train_fit = y_train_improved.loc[train_ok_mask].copy()
X_val_fit = X_val_improved.loc[val_ok_mask].copy()
y_val_fit = y_val_improved.loc[val_ok_mask].copy()

print("Train rows (full -> fit):", len(X_train_improved), "->", len(X_train_fit))
print("Val rows (full -> fit):", len(X_val_improved), "->", len(X_val_fit))
print("X_train_fit:", X_train_fit.shape, "| y_train_fit:", y_train_fit.shape)
print("X_val_fit:", X_val_fit.shape, "| y_val_fit:", y_val_fit.shape)

try:
    improved_cat_model.fit(
        X_train_fit,
        y_train_fit,
        cat_features=improved_cat_feature_indices,
        eval_set=(X_val_fit, y_val_fit),
        use_best_model=True,
    )
except CatBoostError as e:
    if "bad allocation" in str(e).lower():
        print("CatBoost bad allocation detected on GPU.")
        print("Try lower depth/iterations or set smaller gpu_ram_part.")
    raise

print("Improved model training completed.")
print("Best iteration:", improved_cat_model.get_best_iteration())
print("Best validation score:", improved_cat_model.get_best_score())

Train rows (full -> fit): 26958553 -> 26958553
Val rows (full -> fit): 5776530 -> 5776530
X_train_fit: (26958553, 14) | y_train_fit: (26958553,)
X_val_fit: (5776530, 14) | y_val_fit: (5776530,)
0:	learn: 0.4158849	test: 0.4159423	best: 0.4159423 (0)	total: 798ms	remaining: 13m 17s
100:	learn: 0.5253769	test: 0.5258942	best: 0.5258942 (100)	total: 43.8s	remaining: 6m 29s
200:	learn: 0.5427933	test: 0.5433053	best: 0.5433053 (200)	total: 1m 26s	remaining: 5m 42s
300:	learn: 0.5488898	test: 0.5490958	best: 0.5491048 (298)	total: 2m 7s	remaining: 4m 55s
400:	learn: 0.5538258	test: 0.5539074	best: 0.5539074 (400)	total: 2m 48s	remaining: 4m 11s
500:	learn: 0.5587300	test: 0.5587262	best: 0.5587262 (500)	total: 3m 29s	remaining: 3m 28s
600:	learn: 0.5620826	test: 0.5620831	best: 0.5620983 (599)	total: 4m 9s	remaining: 2m 45s
700:	learn: 0.5648284	test: 0.5647607	best: 0.5647607 (700)	total: 4m 50s	remaining: 2m 3s
800:	learn: 0.5669236	test: 0.5668918	best: 0.5668918 (800)	total: 5m 31s	rema

### 11.6 Оценка improved модели

In [38]:
# 11.6 Evaluation block (run after improved model training)
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

if "improved_cat_model" not in globals() or not improved_cat_model.is_fitted():
    print("improved_cat_model is not trained yet. Run the training cell in 11.5 first.")
else:
    val_pred_improved = np.asarray(improved_cat_model.predict(X_val_improved)).reshape(-1).astype(int)
    test_pred_improved = np.asarray(improved_cat_model.predict(X_test_improved)).reshape(-1).astype(int)

    metrics_improved = pd.DataFrame(
        [
            {
                "split": "validation",
                "accuracy": float(accuracy_score(y_val_improved, val_pred_improved)),
                "macro_f1": float(f1_score(y_val_improved, val_pred_improved, average="macro")),
                "weighted_f1": float(f1_score(y_val_improved, val_pred_improved, average="weighted")),
            },
            {
                "split": "test",
                "accuracy": float(accuracy_score(y_test_improved, test_pred_improved)),
                "macro_f1": float(f1_score(y_test_improved, test_pred_improved, average="macro")),
                "weighted_f1": float(f1_score(y_test_improved, test_pred_improved, average="weighted")),
            },
        ]
    )
    print("Improved model metrics:")
    display(metrics_improved)

    labels_sorted_improved = sorted(id_to_target_improved.keys())
    target_names_improved = [id_to_target_improved[label] for label in labels_sorted_improved]

    print("Classification report (validation):")
    print(
        classification_report(
            y_val_improved,
            val_pred_improved,
            labels=labels_sorted_improved,
            target_names=target_names_improved,
            zero_division=0,
        )
    )

    print("Classification report (test):")
    print(
        classification_report(
            y_test_improved,
            test_pred_improved,
            labels=labels_sorted_improved,
            target_names=target_names_improved,
            zero_division=0,
        )
    )

    cm_test_improved = confusion_matrix(
        y_test_improved, test_pred_improved, labels=labels_sorted_improved
    )
    cm_test_improved_df = pd.DataFrame(
        cm_test_improved,
        index=[f"true_{id_to_target_improved[i]}" for i in labels_sorted_improved],
        columns=[f"pred_{id_to_target_improved[i]}" for i in labels_sorted_improved],
    )
    print("Confusion matrix (test):")
    display(cm_test_improved_df)

Improved model metrics:


,split,accuracy,macro_f1,weighted_f1
0,validation,0.699584,0.570208,0.690378
1,test,0.699255,0.569716,0.690047


Classification report (validation):
               precision    recall  f1-score   support

         corn       0.69      0.72      0.71   1761473
       cotton       0.72      0.77      0.75    278886
       fallow       0.59      0.41      0.49    271856
   forage_hay       0.81      0.85      0.83    559815
      legumes       0.50      0.14      0.22     86986
other_cereals       0.53      0.24      0.33    150459
      sorghum       0.50      0.31      0.38    174736
     soybeans       0.72      0.76      0.74   1642055
        wheat       0.66      0.73      0.69    850264

     accuracy                           0.70   5776530
    macro avg       0.64      0.55      0.57   5776530
 weighted avg       0.69      0.70      0.69   5776530

Classification report (test):
               precision    recall  f1-score   support

         corn       0.69      0.72      0.71   1762966
       cotton       0.72      0.77      0.75    279891
       fallow       0.59      0.41      0.48    26

,pred_corn,pred_cotton,pred_fallow,pred_forage_hay,pred_legumes,pred_other_cereals,pred_sorghum,pred_soybeans,pred_wheat
true_corn,1265708,26911,17972,58677,2728,6001,12598,317686,54685
true_cotton,16198,215605,2341,1592,25,367,6920,17780,19063
true_fallow,33959,5432,111120,4151,926,4237,11603,45063,53002
true_forage_hay,43111,1723,2365,476411,520,5784,643,13953,15558
true_legumes,12407,309,1930,2421,12077,1972,173,15580,40418
true_other_cereals,29335,2415,9527,12419,512,37470,2907,11170,45744
true_sorghum,29677,14070,18101,2345,52,2280,53012,9755,44738
true_soybeans,305693,14146,4891,12868,3663,928,3068,1249019,48158
true_wheat,88880,18081,21035,17641,3911,11223,16613,52511,618564


### 11.7 Сохранение improved модели и артефактов
Этот блок запускается после обучения и сохраняет модель, список признаков и mapping классов в единую структуру artifacts/catboost.

In [ ]:
# 11.7 Save improved model and metadata (run after training)
from pathlib import Path
import json

if "improved_cat_model" not in globals() or not improved_cat_model.is_fitted():
    print("improved_cat_model is not trained yet. Train the model before saving.")
else:
    improved_artifacts_dir = Path("artifacts") / "catboost" / "improved"
    improved_artifacts_dir.mkdir(parents=True, exist_ok=True)

    improved_model_path = improved_artifacts_dir / "model.cbm"
    improved_meta_path = improved_artifacts_dir / "meta.json"

    improved_cat_model.save_model(improved_model_path.as_posix())
    print("Improved model saved to:", improved_model_path.resolve())

    improved_meta = {
        "feature_cols": improved_feature_cols,
        "cat_features": improved_cat_features,
        "num_features": improved_num_features,
        "target_column": "target",
        "encoded_target_column": improved_target_col,
        "target_to_id": {str(k): int(v) for k, v in target_to_id_improved.items()},
        "id_to_target": {str(int(k)): str(v) for k, v in id_to_target_improved.items()},
        "split_source": split_source,
    }

    with open(improved_meta_path, "w", encoding="utf-8") as f:
        json.dump(improved_meta, f, ensure_ascii=False, indent=2)

    print("Improved metadata saved to:", improved_meta_path.resolve())

Improved model saved to: C:\Users\Dmitry\code-projects\diploma-crop-rotation\models\catboost_improved_features.cbm
Improved metadata saved to: C:\Users\Dmitry\code-projects\diploma-crop-rotation\models\catboost_improved_features_meta.json


: 